# Aura Health — LangGraph Multi-Agent Consultation System

**Stack:** LangGraph `StateGraph` · AWS Bedrock (Claude) · FAISS · presidio PII scrubber · HuggingFace Med42 · IMDA Governance · Optional OpenAI API STT · Optional FastAPI frontend bridge

---

## Architecture

The clinical pipeline now supports optional speech-to-text preprocessing before intake and an optional schema-first frontend bridge after graph compilation.

```
START
  └─► stt_prep_node            (marks transcript source: manual or OpenAI mic STT)
        └─► intake_node        (PII scrub via presidio)
              └─► supervisor_node    (routes to one or more specialist agents)
                    ├─► clinical_node    (CDC guideline RAG + Med42 fallback on RAG miss)
                    ├─► drug_node        (drug interaction check)
                    └─► research_node    (live PubMed retrieval)
                          └─► summary_node   (synthesises SOAP note)
                                    └─► [Governance postpass]
                                              └─► END
```

---

## Governance

After `summary_node`, every consultation passes through a sequential IMDA Model AI Governance Framework postpass.

```
summary_node
  └─► xai_node                 (explainability — evidence sources & confidence)
        └─► fairness_node      (demographic bias detection)
              └─► human_oversight_node    (HITL escalation flags)
                    └─► clinical_safety_node  (contraindication / red-flag guard)
                              └─► audit_node  (immutable JSONL audit log)
                                        └─► END
```

| Node | Module | IMDA Principle |
|------|--------|----------------|
| `xai_node` | `governance/xai_layer.py` | Transparency |
| `fairness_node` | `governance/fairness_monitor.py` | Fairness |
| `human_oversight_node` | `governance/human_oversight.py` | Human control |
| `clinical_safety_node` | `governance/clinical_safety_guard.py` | Safety |
| `audit_node` | `governance/audit_log.py` | Accountability |

---

## Optional STT Input (Cells 10a and 10b)

- **Cell 10a (mic mode):** set `USE_STT_MIC_DEFAULT=true` in `.env` to auto-enable OpenAI API mic transcription each run.
- **Cell 10a model default:** `STT_OPENAI_MODEL=gpt-4o-mini`.
- **Cell 10b (smoke mode):** set `STT_SMOKE_TEST=True` to inject sample STT text without microphone recording.
- Cell 11 automatically falls back to manual transcript if STT text is empty.


---

## Scenario Demos (Cell 12a)

Two self-contained runnable demos are embedded at **Cell 12a**.

| Scenario | What it demonstrates |
|----------|----------------------|
| **A — RAG miss → Med42 fallback** | When lexical overlap with retrieved chunks < 0.08, `clinical_node` calls `m42-health/Llama3-Med42-8B` via HuggingFace Inference API as a secondary advisory. |
| **B — Live PubMed research with multi-tier fallback** | Try live PubMed first, then Med42, then Bedrock as last resort if upstream retrieval/checkers fail. |

---

## Run Order

| Step | Cell | Purpose |
|------|------|---------|
| 1 | 1 | Install dependencies |
| 2 | 9c | Build FAISS knowledge base |
| 3 | 9 | Compile LangGraph pipeline |
| 4 | 10a | Optional mic STT transcription |
| 5 | 10b | Optional STT smoke test (no mic) |
| 6 | 11 | Run full consultation |
| 7 | 12 | Display SOAP note |
| 8 | 12a | Scenario demos (standalone) |
| 9 | 12b | Governance report |
| 10 | 17 | FastAPI bridge workflow and API server |
| 11 | 18 | Frontend schema-first PHP and JavaScript examples |

---

## Frontend Bridge Quick Reference

Use the FastAPI bridge when a frontend needs to set patient context dynamically instead of editing the notebook.

- `GET /consult/schema` returns field metadata, defaults, example payloads, and frontend integration hints.
- `POST /consult` accepts `session_id`, `transcript`, and `patient_context`.
- `GET /stream/{session_id}` returns incremental node output.
- `GET /session/{session_id}` returns `status`, effective `patient_context`, and the final SOAP/governance payload.
- Session status values are `queued`, `running`, `completed`, and `failed`.

---

## Prerequisites
```bash
pip install langgraph langchain langchain-aws langchain-community \
            boto3 faiss-cpu sentence-transformers \
            presidio-analyzer presidio-anonymizer \
            langchain-huggingface huggingface-hub openai \
            nest-asyncio ipywidgets python-dotenv \
            fastapi uvicorn sse-starlette
python -m spacy download en_core_web_lg
```

## AWS Setup
```bash
aws configure   # set access key, secret, region (ap-southeast-1)
# Then enable Claude models in AWS Console -> Bedrock -> Model access
```

---
## Cell 1 — Install dependencies
Run once. Safe to skip if already installed.

In [ ]:
import subprocess, sys, importlib.metadata

def _installed(spec: str) -> bool:
    """Check if a distribution is already installed (ignores version pinning)."""
    name = spec.split(">=")[0].split("==")[0].split("[")[0]
    try:
        importlib.metadata.version(name)
        return True
    except importlib.metadata.PackageNotFoundError:
        return False

packages = [
    "langgraph>=0.2.0",
    "langchain>=0.3.0",
    "langchain-aws>=0.2.0",
    "langchain-community>=0.3.0",
    "langchain-huggingface",
    "huggingface-hub",
    "openai",
    "boto3",
    "faiss-cpu",
    "sentence-transformers",
    "presidio-analyzer",
    "presidio-anonymizer",
    "nest-asyncio",
    "ipywidgets",
    "python-dotenv",
]

missing = [p for p in packages if not _installed(p)]
if missing:
    print(f"Installing {len(missing)} missing package(s): {', '.join(missing)}")
    subprocess.run([sys.executable, "-m", "pip", "install"] + missing + ["-q"], check=True)
    print("Packages installed.")
else:
    print("All packages already satisfied — skipping pip install.")

# spaCy model — only download if not already present
try:
    import spacy
    spacy.load("en_core_web_lg")
    print("spaCy model en_core_web_lg already present.")
except OSError:
    print("Downloading spaCy model en_core_web_lg...")
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_lg", "-q"])
    print("spaCy model ready.")

print("\nSetup complete.")

---
## Cell 2 — Imports & Configuration

In [ ]:
import os
import json
import boto3
import operator
import warnings
import nest_asyncio
from typing import TypedDict, Annotated, List, Literal
from datetime import datetime

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# LangChain
from langchain_aws import ChatBedrock
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# PII scrubbing
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

from dotenv import load_dotenv
import os

load_dotenv()  # reads .env from current folder

warnings.filterwarnings("ignore")
nest_asyncio.apply()  # allows asyncio inside Jupyter

# -- AWS Configuration ----------------------------------------------------------
AWS_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
BEDROCK_MODEL = os.getenv("BEDROCK_MODEL", "anthropic.claude-haiku-4-5-20251001-v1:0")
BEDROCK_INFERENCE_PROFILE_ID = os.getenv("BEDROCK_INFERENCE_PROFILE_ID", "").strip()
BEDROCK_PROVIDER = os.getenv("BEDROCK_PROVIDER", "anthropic").strip()

# -- Fallback: direct Anthropic API (uncomment if not using Bedrock) -----------
# from langchain_anthropic import ChatAnthropic
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

print("Configuration loaded.")
print(f"  Region   : {AWS_REGION}")
print(f"  Model    : {BEDROCK_MODEL}")
if BEDROCK_INFERENCE_PROFILE_ID:
    print(f"  Profile  : {BEDROCK_INFERENCE_PROFILE_ID}")
else:
    print("  Profile  : (not set)")
print(f"  Provider : {BEDROCK_PROVIDER}")
print(f"  Time     : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## Cell 3 — Verify AWS Credentials & Bedrock Access

In [ ]:
def check_aws():
    """Verify credentials and Bedrock model availability."""
    try:
        sts = boto3.client("sts", region_name=AWS_REGION)
        identity = sts.get_caller_identity()
        print(f"AWS identity verified")
        print(f"  Account : {identity['Account']}")
        print(f"  ARN     : {identity['Arn']}")
    except Exception as e:
        print(f"AWS credentials error: {e}")
        print("Run: aws configure")
        return False

    try:
        bedrock = boto3.client("bedrock", region_name=AWS_REGION)
        models = bedrock.list_foundation_models(byProvider="Anthropic")
        available = [m["modelId"] for m in models["modelSummaries"]
                     if m["modelLifecycle"]["status"] == "ACTIVE"]
        print(f"\nActive Anthropic models in {AWS_REGION}:")
        for m in available:
            marker = " <-- selected" if m == BEDROCK_MODEL else ""
            print(f"  {m}{marker}")
        if BEDROCK_MODEL not in available:
            print(f"\nWARNING: {BEDROCK_MODEL} not active.")
            print("Go to AWS Console → Bedrock → Model access → Enable Claude models")
            return False
        return True
    except Exception as e:
        print(f"Bedrock check error: {e}")
        return False

aws_ok = check_aws()

---
## Cell 4 — LLM Initialisation (Bedrock or Anthropic fallback)

In [ ]:
def build_llm():
    """Build LLM — tries Bedrock first, falls back to direct Anthropic."""
    if aws_ok:
        session = boto3.Session(region_name=AWS_REGION)
        client = session.client("bedrock-runtime")

        # Some models cannot be invoked on-demand and require an inference profile ID/ARN.
        runtime_model_id = BEDROCK_INFERENCE_PROFILE_ID or BEDROCK_MODEL

        bedrock_kwargs = {
            "model_id": runtime_model_id,
            "client": client,
            "model_kwargs": {
                "max_tokens": 2048,
                "temperature": 0.1,
                "anthropic_version": "bedrock-2023-05-31"
            },
        }

        # When model_id is an ARN (for profile/provisioned throughput), provider must be explicit.
        if runtime_model_id.startswith("arn:"):
            bedrock_kwargs["provider"] = BEDROCK_PROVIDER

        llm = ChatBedrock(**bedrock_kwargs)

        if BEDROCK_INFERENCE_PROFILE_ID:
            print(f"LLM: AWS Bedrock via inference profile ({BEDROCK_INFERENCE_PROFILE_ID})")
        else:
            print(f"LLM: AWS Bedrock ({BEDROCK_MODEL})")
            print("Tip: set BEDROCK_INFERENCE_PROFILE_ID if your model requires profile-based invocation.")
    else:
        # Direct Anthropic fallback
        from langchain_anthropic import ChatAnthropic
        api_key = os.getenv("ANTHROPIC_API_KEY", "")
        if not api_key:
            raise ValueError("Set ANTHROPIC_API_KEY env var or fix AWS credentials")
        llm = ChatAnthropic(
            model="claude-haiku-4-5-20251001",
            api_key=api_key,
            max_tokens=2048,
            temperature=0.1
        )
        print("LLM: Direct Anthropic API (claude-haiku-4-5-20251001)")

    # Quick smoke test
    try:
        test = llm.invoke("Reply with exactly: AURA_READY")
    except Exception as e:
        msg = str(e)
        if "inference profile" in msg.lower() and not BEDROCK_INFERENCE_PROFILE_ID:
            raise ValueError(
                "This model requires an inference profile. Set BEDROCK_INFERENCE_PROFILE_ID to a valid profile ID or ARN, then rerun Cells 2-4."
            ) from e
        if "model provider should be supplied" in msg.lower() and (BEDROCK_INFERENCE_PROFILE_ID or BEDROCK_MODEL).startswith("arn:"):
            raise ValueError(
                "Set BEDROCK_PROVIDER in .env (for example: BEDROCK_PROVIDER=anthropic), then rerun Cells 2-4."
            ) from e
        raise

    if "AURA_READY" in test.content:
        print("LLM smoke test: PASSED")
    else:
        print(f"LLM smoke test unexpected response: {test.content[:80]}")
    return llm

llm = build_llm()

---
## Cell 5 — PII Scrubber (presidio, HIPAA Safe Harbor + Singapore context)
This step removes personally identifiable information (PII) from raw transcript text before any downstream summarization or retrieval.

How it works (brief):
- **Presidio Analyzer** detects PII entities and returns spans with confidence scores.
- **spaCy model (`en_core_web_lg`)** powers the NLP layer used by Presidio for tokenization, linguistic parsing, and NER context.
- **Singapore extensions** add regex-based recognizers for:
  - **NRIC/FIN** patterns (for example `S1234567D`, `F7654321N`)
  - **Local address patterns** (for example `Blk 123 ... #04-55 Singapore 120123`)
- **Presidio Anonymizer** replaces each detected span using configured operators, mapping to readable tags such as `[PATIENT]`, `[SG_NRIC]`, `[SG_ADDRESS]`, and `[PHONE]`.

Outcome:
- The pipeline preserves clinical meaning while removing direct identifiers, including Singapore-specific patient identifiers.

In [ ]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer
from presidio_analyzer.pattern import Pattern
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

# Singapore-specific recognizers
nric_recognizer = PatternRecognizer(
    supported_entity="SG_NRIC",
    supported_language="en",
    patterns=[
        Pattern(
            name="sg_nric_fin",
            regex=r"\b[STFGM]\d{7}[A-Z]\b",
            score=0.85,
        )
    ],
)

sg_address_recognizer = PatternRecognizer(
    supported_entity="SG_ADDRESS",
    supported_language="en",
    patterns=[
        Pattern(
            name="sg_block_unit_postal",
            regex=r"\b(?:blk|block)\s*\d+[A-Za-z]?\s+[A-Za-z0-9'./\-\s]{3,80}#?\d{1,3}-\d{1,3}(?:\s+singapore\s+\d{6})?\b",
            score=0.7,
        ),
        Pattern(
            name="sg_street_postal",
            regex=r"\b\d{1,4}\s+[A-Za-z0-9'./\-\s]{3,80}\s+singapore\s+\d{6}\b",
            score=0.65,
        ),
    ],
)

analyzer.registry.add_recognizer(nric_recognizer)
analyzer.registry.add_recognizer(sg_address_recognizer)

# Entity types to scrub — HIPAA Safe Harbor identifiers + Singapore-specific entities
SCRUB_ENTITIES = [
    "PERSON",
    "PHONE_NUMBER",
    "EMAIL_ADDRESS",
    "LOCATION",
    "DATE_TIME",
    "MEDICAL_LICENSE",
    "URL",
    "IP_ADDRESS",
    "SG_NRIC",
    "SG_ADDRESS",
]

# Replace each entity type with a descriptive tag rather than [ANONYMIZED]
OPERATORS = {
    "PERSON": OperatorConfig("replace", {"new_value": "[PATIENT]"}),
    "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "[PHONE]"}),
    "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "[EMAIL]"}),
    "LOCATION": OperatorConfig("replace", {"new_value": "[LOCATION]"}),
    "DATE_TIME": OperatorConfig("replace", {"new_value": "[DATE]"}),
    "SG_NRIC": OperatorConfig("replace", {"new_value": "[SG_NRIC]"}),
    "SG_ADDRESS": OperatorConfig("replace", {"new_value": "[SG_ADDRESS]"}),
    "DEFAULT": OperatorConfig("replace", {"new_value": "[REDACTED]"}),
}

ENTITY_PRIORITY = {
    "SG_NRIC": 100,
    "SG_ADDRESS": 95,
    "PHONE_NUMBER": 90,
    "EMAIL_ADDRESS": 85,
    "PERSON": 80,
    "MEDICAL_LICENSE": 75,
    "LOCATION": 70,
    "DATE_TIME": 50,
    "URL": 40,
    "IP_ADDRESS": 40,
}


def _is_better(a, b) -> bool:
    a_key = (
        ENTITY_PRIORITY.get(a.entity_type, 10),
        round(float(a.score), 4),
        a.end - a.start,
    )
    b_key = (
        ENTITY_PRIORITY.get(b.entity_type, 10),
        round(float(b.score), 4),
        b.end - b.start,
    )
    return a_key > b_key


def _resolve_overlaps(results: list) -> list:
    selected = []
    for r in sorted(results, key=lambda x: (x.start, x.end)):
        overlaps = [
            i
            for i, s in enumerate(selected)
            if not (r.end <= s.start or r.start >= s.end)
        ]
        if not overlaps:
            selected.append(r)
            continue

        if all(_is_better(r, selected[i]) for i in overlaps):
            for i in reversed(overlaps):
                selected.pop(i)
            selected.append(r)

    return sorted(selected, key=lambda x: x.start)


def scrub_pii(text: str) -> tuple[str, list]:
    """Returns (scrubbed_text, list_of_detected_entities)."""
    raw_results = analyzer.analyze(
        text=text,
        entities=SCRUB_ENTITIES,
        language="en",
    )
    results = _resolve_overlaps(raw_results)
    scrubbed = anonymizer.anonymize(
        text=text,
        analyzer_results=results,
        operators=OPERATORS,
    ).text
    detected = [{"type": r.entity_type, "score": round(r.score, 2)} for r in results]
    return scrubbed, detected


# Test the scrubber
test_text = (
    "Patient John Smith NRIC S1234567D, DOB 12/03/1980, called from +65-9123-4567 "
    "about chest pain. Email: john.smith@example.com. "
    "Address: Blk 123 Queenstown Ave #04-55 Singapore 120123."
)
scrubbed, entities = scrub_pii(test_text)
print("PII Scrubber test")
print(f"  Input   : {test_text}")
print(f"  Scrubbed: {scrubbed}")
print(f"  Detected: {entities}")

---
## Cell 6 — Hybrid knowledge base (seed + optional SG guideline/PubMed/openFDA ingestion)
This section builds the retrieval corpus used by the FAISS vector store. It combines a small curated seed set with optional live and local external sources so the clinical, drug, and research agents can retrieve grounded evidence before generating outputs.

What is included:
- **Seed documents** for fast baseline coverage of common primary-care and medication scenarios.
- **Singapore-relevant guidance pages** (for example MOH/LIA URLs) for local policy and health guidance context.
- **PubMed abstracts** for recent medical literature and guideline-related evidence.
- **openFDA drug label API** for structured medication safety content such as warnings, contraindications, and boxed warnings.
- **openFDA drug adverse event API** for real-world reported side effects and seriousness flags.
- **Optional local FDA ZIP ingestion** from `data/fda_zip_drop` when you want offline or bulk label ingestion.

Important behavior:
- Adverse event records are **compressed/deduplicated** before indexing so repeated reaction patterns do not flood the vector store.
- Documents are **chunked with overlap** before embedding so retrieval can match symptom-specific passages rather than only whole long documents.
- The resulting chunks are embedded and stored in **FAISS**, then retrieved later by semantic similarity during agent execution.
- Multiple sources can be enabled together, so the retriever can return both regulatory safety guidance and research evidence in the same run.

Production note:
- SG guideline pages (MOH/LIA): free, no API key required
- PubMed E-utilities: free, API key optional for higher rate limits
- openFDA drug labels API: free, API key optional
- openFDA drug adverse event API: free, API key optional
- FDA local ZIP dataset: no API dependency once downloaded

Useful `.env` keys:
- `KB_ENABLE_OPENFDA=true`
- `KB_ENABLE_OPENFDA_EVENT=true`
- `KB_ENABLE_OPENFDA_ZIP=true`
- `OPENFDA_TERMS=lisinopril,metformin,ibuprofen`
- `OPENFDA_EVENT_TERMS=lisinopril,metformin,ibuprofen`
- `OPENFDA_EVENT_PER_TERM_LIMIT=5`
- `RAG_CHUNK_SIZE=900`
- `RAG_CHUNK_OVERLAP=150`
- `OPENFDA_ZIP_DIR=data/fda_zip_drop`
- `OPENFDA_ZIP_MAX_DOCS=200`

In [ ]:
import os
import re
import html
import json
import glob
import zipfile
from pathlib import Path
from dotenv import load_dotenv
from urllib.parse import urlencode, quote_plus
from urllib.request import Request, urlopen
import xml.etree.ElementTree as ET

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Loading embedding model (first run downloads ~90MB)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)
load_dotenv(override=True)

# -- Config from .env ---------------------------------------------------------
KB_USE_SEED = os.getenv("KB_USE_SEED", "true").lower() == "true"
KB_ENABLE_CDC = os.getenv("KB_ENABLE_CDC", "false").lower() == "true"
KB_ENABLE_PUBMED = os.getenv("KB_ENABLE_PUBMED", "false").lower() == "true"
KB_ENABLE_OPENFDA = os.getenv("KB_ENABLE_OPENFDA", "false").lower() == "true"
KB_ENABLE_OPENFDA_ZIP = os.getenv("KB_ENABLE_OPENFDA_ZIP", "false").lower() == "true"
KB_ENABLE_OPENFDA_EVENT = os.getenv("KB_ENABLE_OPENFDA_EVENT", "false").lower() == "true"

RAG_CHUNK_SIZE = int(os.getenv("RAG_CHUNK_SIZE", "900"))
RAG_CHUNK_OVERLAP = int(os.getenv("RAG_CHUNK_OVERLAP", "150"))

CDC_GUIDELINE_URLS = [u.strip() for u in os.getenv("CDC_GUIDELINE_URLS", "https://www.cdc.gov/high-blood-pressure/about/index.html,https://www.cdc.gov/heart-disease/about/index.html").split(",") if u.strip()]

PUBMED_QUERY    = os.getenv("PUBMED_QUERY", "hypertension guideline OR chest pain triage OR type 2 diabetes management")
PUBMED_RETMAX   = int(os.getenv("PUBMED_RETMAX", "8"))
PUBMED_EMAIL    = os.getenv("PUBMED_EMAIL", "")
PUBMED_TOOL     = os.getenv("PUBMED_TOOL", "aura-health-rag")
NCBI_API_KEY    = os.getenv("NCBI_API_KEY", "")

OPENFDA_TERMS   = [t.strip() for t in os.getenv("OPENFDA_TERMS", "lisinopril,metformin,ibuprofen").split(",") if t.strip()]
OPENFDA_API_KEY = os.getenv("OPENFDA_API_KEY", "")
OPENFDA_ZIP_DIR = os.getenv("OPENFDA_ZIP_DIR", "data/fda_zip_drop")

OPENFDA_EVENT_TERMS = [t.strip() for t in os.getenv("OPENFDA_EVENT_TERMS", "lisinopril,metformin,ibuprofen").split(",") if t.strip()]
OPENFDA_EVENT_PER_TERM_LIMIT = int(os.getenv("OPENFDA_EVENT_PER_TERM_LIMIT", "5"))
OPENFDA_ZIP_MAX_DOCS = int(os.getenv("OPENFDA_ZIP_MAX_DOCS", "200"))

# -- Helper text cleaning -------------------------------------------------------

def clean_text(raw: str, max_chars: int = 2500) -> str:
    raw = re.sub(r"<script[\s\S]*?</script>", " ", raw, flags=re.IGNORECASE)
    raw = re.sub(r"<style[\s\S]*?</style>",   " ", raw, flags=re.IGNORECASE)
    raw = re.sub(r"<[^>]+>", " ", raw)
    raw = html.unescape(raw)
    raw = re.sub(r"\s+", " ", raw).strip()
    return raw[:max_chars]

# -- Seed documents (17 total: covers common illness, cardiology, emergency, etc.) ----

def seed_docs() -> list:
    return [
        Document(page_content="Hypertension Stage 1: systolic 130-139 or diastolic 80-89 mmHg. First-line treatment includes lifestyle changes and single-drug therapy based on patient profile.", metadata={"source": "Seed: ACC/AHA style", "category": "cardiology"}),
        Document(page_content="Hypertension Stage 2: systolic >=140 or diastolic >=90 mmHg. Combination therapy is often needed and renal/electrolyte monitoring is recommended.", metadata={"source": "Seed: JNC style", "category": "cardiology"}),
        Document(page_content="Chest tightness differential includes ACS, GERD, musculoskeletal pain, and anxiety. Initial workup can include ECG, troponin, and chest radiography.", metadata={"source": "Seed: Emergency triage", "category": "emergency"}),
        Document(page_content="Lisinopril: ACE inhibitor for hypertension. Monitor creatinine and potassium. Common adverse effect is cough. Contraindicated in pregnancy.", metadata={"source": "Seed: Drug reference", "category": "pharmacology"}),
        Document(page_content="Metformin: first-line for type 2 diabetes. Avoid in severe renal dysfunction. Review around iodinated contrast procedures.", metadata={"source": "Seed: ADA style", "category": "endocrinology"}),
        Document(page_content="Type 2 diabetes HbA1c targets are commonly <7 percent for many adults, with individualized goals based on comorbidity and frailty.", metadata={"source": "Seed: ADA style", "category": "endocrinology"}),
        Document(page_content="Drug interaction: ACE inhibitors and NSAIDs can reduce antihypertensive efficacy and raise risk of kidney injury in susceptible patients.", metadata={"source": "Seed: Interaction", "category": "drug_interaction"}),
        Document(page_content="Shortness of breath workup may include pulse oximetry, respiratory rate, chest x-ray, and consideration of PE, CHF, COPD, and pneumonia.", metadata={"source": "Seed: Respiratory triage", "category": "respiratory"}),
        Document(page_content="Common cold (viral URI) usually includes runny nose, nasal congestion, sore throat, cough, and low-grade fever. Supportive care includes fluids, rest, and symptomatic relief. Antibiotics are not indicated for uncomplicated viral colds.", metadata={"source": "Seed: Primary care URI", "category": "common_illness"}),
        Document(page_content="Influenza-like illness often presents with acute fever, myalgia, headache, fatigue, cough, and sore throat. Higher-risk patients (older adults, pregnancy, chronic disease, immunocompromise) may benefit from early antiviral evaluation.", metadata={"source": "Seed: Influenza triage", "category": "common_illness"}),
        Document(page_content="Fever in adults is commonly defined as temperature >=38.0 C. Initial assessment includes symptom duration, exposure history, hydration status, and red flags such as confusion, persistent vomiting, severe shortness of breath, or hypotension.", metadata={"source": "Seed: Fever assessment", "category": "common_illness"}),
        Document(page_content="Acute uncomplicated headache can be tension-type or migraine. Red flags that require urgent in-person evaluation include thunderclap onset, focal neurologic deficits, fever with neck stiffness, head trauma, altered mental status, or new headache in older age.", metadata={"source": "Seed: Headache triage", "category": "neurology"}),
        Document(page_content="Acute diarrhea is often viral and self-limited. Focus on oral rehydration and monitoring for dehydration. Escalate care for blood in stool, persistent high fever, severe abdominal pain, signs of dehydration, or symptoms lasting beyond several days.", metadata={"source": "Seed: GI triage", "category": "gastrointestinal"}),
        Document(page_content="Nausea and vomiting management prioritizes hydration and electrolyte balance. Warning signs include inability to keep fluids down, bilious or bloody emesis, severe abdominal pain, pregnancy-related dehydration risk, and symptoms of metabolic disturbance.", metadata={"source": "Seed: GI symptom care", "category": "gastrointestinal"}),
        Document(page_content="Sore throat is commonly viral. Consider streptococcal pharyngitis when fever, tonsillar exudate, and tender anterior cervical nodes are present without cough. Immediate evaluation is needed for airway compromise, drooling, muffled voice, or unilateral neck swelling.", metadata={"source": "Seed: ENT triage", "category": "common_illness"}),
        Document(page_content="Acute cough with cold symptoms is usually viral bronchitis or URI. Evaluate urgently when there is chest pain, hemoptysis, oxygen desaturation, severe dyspnea, or persistent fever suggesting pneumonia or another serious cause.", metadata={"source": "Seed: Cough triage", "category": "respiratory"}),
        Document(page_content="General safety: emergency red flags across common illness include severe chest pain, severe shortness of breath, confusion, seizure, syncope, unilateral weakness, cyanosis, uncontrolled bleeding, or rapidly worsening symptoms.", metadata={"source": "Seed: Universal red flags", "category": "clinical_safety"}),
    ]

# -- Optional fetchers (CDC, PubMed, openFDA, ZIP) ---------------------------------

def fetch_cdc_docs(urls: list) -> list:
    docs = []
    for url in urls:
        try:
            req = Request(url, headers={"User-Agent": "Mozilla/5.0 aura-health-rag"})
            with urlopen(req, timeout=25) as r:
                html_text = r.read().decode("utf-8", errors="ignore")
            text = clean_text(html_text, max_chars=3500)
            if len(text) > 300:
                docs.append(Document(page_content=text, metadata={"source": url, "category": "cdc"}))
        except Exception as e:
            print(f"CDC fetch skipped ({url}): {e}")
    return docs


def fetch_pubmed_docs(query: str, retmax: int = 8) -> list:
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
    esearch_params = {"db": "pubmed", "term": query, "retmode": "json", "retmax": str(retmax), "tool": PUBMED_TOOL}
    if PUBMED_EMAIL:  esearch_params["email"] = PUBMED_EMAIL
    if NCBI_API_KEY:  esearch_params["api_key"] = NCBI_API_KEY

    search_url = f"{base}/esearch.fcgi?{urlencode(esearch_params)}"
    with urlopen(search_url, timeout=25) as r:
        search_json = json.loads(r.read().decode("utf-8", errors="ignore"))

    ids = search_json.get("esearchresult", {}).get("idlist", [])
    if not ids:
        return []

    efetch_params = {"db": "pubmed", "id": ",".join(ids), "retmode": "xml", "tool": PUBMED_TOOL}
    if PUBMED_EMAIL:  efetch_params["email"] = PUBMED_EMAIL
    if NCBI_API_KEY:  efetch_params["api_key"] = NCBI_API_KEY

    fetch_url = f"{base}/efetch.fcgi?{urlencode(efetch_params)}"
    with urlopen(fetch_url, timeout=25) as r:
        xml_data = r.read().decode("utf-8", errors="ignore")

    root = ET.fromstring(xml_data)
    docs = []
    for article in root.findall(".//PubmedArticle"):
        title_node     = article.find(".//ArticleTitle")
        abstract_nodes = article.findall(".//Abstract/AbstractText")
        pmid_node      = article.find(".//PMID")
        title    = "".join(title_node.itertext()).strip() if title_node is not None else "Untitled"
        abstract = " ".join("".join(n.itertext()).strip() for n in abstract_nodes if n is not None).strip()
        pmid     = pmid_node.text.strip() if pmid_node is not None and pmid_node.text else "unknown"
        if abstract:
            docs.append(Document(
                page_content=clean_text(f"Title: {title}. Abstract: {abstract}", max_chars=3500),
                metadata={"source": f"PubMed:{pmid}", "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/", "category": "research"},
            ))
    return docs


def fetch_openfda_docs(terms: list, per_term_limit: int = 1) -> list:
    docs = []
    for term in terms:
        try:
            query = f'openfda.generic_name:"{term}"'
            url = f"https://api.fda.gov/drug/label.json?search={quote_plus(query)}&limit={per_term_limit}"
            if OPENFDA_API_KEY:
                url += f"&api_key={quote_plus(OPENFDA_API_KEY)}"
            with urlopen(url, timeout=25) as r:
                payload = json.loads(r.read().decode("utf-8", errors="ignore"))
            for item in payload.get("results", []):
                warnings_text   = " ".join(item.get("warnings", [])[:2])
                boxed           = " ".join(item.get("boxed_warning", [])[:1])
                contraindications = " ".join(item.get("contraindications", [])[:2])
                safety_text = " ".join([warnings_text, boxed, contraindications]).strip()
                if safety_text:
                    docs.append(Document(
                        page_content=clean_text(f"Drug: {term}. Safety notes: {safety_text}", max_chars=2500),
                        metadata={"source": f"openFDA:{term}", "category": "drug_safety"},
                    ))
        except Exception as e:
            print(f"openFDA API fetch skipped ({term}): {e}")
    return docs


def _event_drug_name(item: dict, fallback: str) -> str:
    patient = item.get("patient", {}) if isinstance(item, dict) else {}
    drugs   = patient.get("drug", []) if isinstance(patient, dict) else []
    names   = [str(d.get("medicinalproduct")) for d in drugs[:3] if isinstance(d, dict) and d.get("medicinalproduct")]
    return " | ".join(names) if names else fallback


def _event_reactions(item: dict) -> str:
    patient   = item.get("patient", {}) if isinstance(item, dict) else {}
    reactions = patient.get("reaction", []) if isinstance(patient, dict) else []
    terms = [str(r.get("reactionmeddrapt")) for r in reactions[:8] if isinstance(r, dict) and r.get("reactionmeddrapt")]
    return ", ".join(terms)


def _norm_token(value: str) -> str:
    return re.sub(r"\s+", " ", (value or "").strip().lower())


def fetch_openfda_adverse_event_docs(terms: list, per_term_limit: int = 5) -> list:
    docs = []
    for term in terms:
        try:
            query = f'patient.drug.medicinalproduct:"{term}"'
            url = f"https://api.fda.gov/drug/event.json?search={quote_plus(query)}&limit={per_term_limit}"
            if OPENFDA_API_KEY:
                url += f"&api_key={quote_plus(OPENFDA_API_KEY)}"
            with urlopen(url, timeout=30) as r:
                payload = json.loads(r.read().decode("utf-8", errors="ignore"))

            groups: dict = {}
            for item in payload.get("results", []):
                if not isinstance(item, dict):
                    continue
                reactions_raw = _event_reactions(item)
                if not reactions_raw:
                    continue
                serious      = "yes" if str(item.get("serious", "0")) == "1" else "no"
                outcome      = []
                if str(item.get("seriousnessdeath",          "0")) == "1": outcome.append("death")
                if str(item.get("seriousnesshospitalization", "0")) == "1": outcome.append("hospitalization")
                if str(item.get("seriousnessdisabling",       "0")) == "1": outcome.append("disabling")
                if str(item.get("seriousnesslifethreatening", "0")) == "1": outcome.append("life-threatening")
                outcome_text = ", ".join(outcome) if outcome else "not specified"
                received     = str(item.get("receiptdate", "unknown"))
                report_id    = str(item.get("safetyreportid", "unknown"))
                drug_name    = _event_drug_name(item, term)
                reaction_terms = sorted(set(_norm_token(t) for t in reactions_raw.split(",") if _norm_token(t)))
                if not reaction_terms:
                    continue
                signature = (_norm_token(term), _norm_token(drug_name), tuple(reaction_terms), serious, _norm_token(outcome_text))
                if signature not in groups:
                    groups[signature] = {"term": term, "drug_name": drug_name, "serious": serious,
                                         "outcome_text": outcome_text, "reactions": ", ".join(reaction_terms),
                                         "count": 0, "sample_report_ids": [], "latest_receipt_date": received}
                g = groups[signature]
                g["count"] += 1
                if len(g["sample_report_ids"]) < 5:
                    g["sample_report_ids"].append(report_id)
                if received > g["latest_receipt_date"]:
                    g["latest_receipt_date"] = received

            for g in groups.values():
                content = (
                    f"Drug adverse event pattern for {g['drug_name']}. "
                    f"Observed in {g['count']} report(s). Serious: {g['serious']}. "
                    f"Outcome flags: {g['outcome_text']}. Reported reactions: {g['reactions']}. "
                    f"Latest receipt date: {g['latest_receipt_date']}."
                )
                docs.append(Document(
                    page_content=clean_text(content, max_chars=2500),
                    metadata={"source": f"openFDA-event:{g['term']}", "category": "drug_adverse_event",
                              "compressed": True, "report_count": g["count"], "sample_report_ids": g["sample_report_ids"]},
                ))
        except Exception as e:
            print(f"openFDA adverse event fetch skipped ({term}): {e}")
    return docs


def _extract_openfda_items(payload) -> list:
    if isinstance(payload, dict) and isinstance(payload.get("results"), list):
        return payload["results"]
    if isinstance(payload, list):
        return payload
    return []


def fetch_openfda_zip_docs(zip_dir: str, max_docs: int = 200) -> list:
    docs = []
    base = Path(zip_dir)
    if not base.exists():
        print(f"FDA ZIP folder not found: {base}")
        return docs
    zip_paths = sorted(glob.glob(str(base / "*.zip")))
    if not zip_paths:
        print(f"No ZIP files found in: {base}")
        return docs
    for zpath in zip_paths:
        if len(docs) >= max_docs:
            break
        try:
            with zipfile.ZipFile(zpath, "r") as zf:
                for member in [n for n in zf.namelist() if n.lower().endswith(".json")]:
                    if len(docs) >= max_docs:
                        break
                    with zf.open(member) as f:
                        try:
                            payload = json.loads(f.read().decode("utf-8", errors="ignore"))
                        except Exception:
                            continue
                    for item in _extract_openfda_items(payload):
                        if len(docs) >= max_docs:
                            break
                        if not isinstance(item, dict):
                            continue
                        openfda      = item.get("openfda", {}) or {}
                        generic_names = openfda.get("generic_name", [])
                        brand_names  = openfda.get("brand_name", [])
                        drug_name    = " | ".join(generic_names[:2]) or " | ".join(brand_names[:2]) or "unknown"
                        warnings_t   = " ".join(item.get("warnings", [])[:2])
                        boxed        = " ".join(item.get("boxed_warning", [])[:1])
                        contraindic  = " ".join(item.get("contraindications", [])[:2])
                        indications  = " ".join(item.get("indications_and_usage", [])[:1])
                        safety_text  = " ".join([warnings_t, boxed, contraindic, indications]).strip()
                        if not safety_text:
                            continue
                        docs.append(Document(
                            page_content=clean_text(f"Drug: {drug_name}. Safety notes: {safety_text}", max_chars=2500),
                            metadata={"source": f"FDA_ZIP:{Path(zpath).name}", "category": "drug_safety_zip", "zip_member": member},
                        ))
        except Exception as e:
            print(f"FDA ZIP ingest skipped ({zpath}): {e}")
    return docs

# ── Build hybrid corpus + chunk + index ────────────────────────────────────────

CLINICAL_DOCS: list = []

if KB_USE_SEED:
    CLINICAL_DOCS.extend(seed_docs())
if KB_ENABLE_CDC:
    print(f"Fetching CDC docs from {len(CDC_GUIDELINE_URLS)} URLs...")
    CLINICAL_DOCS.extend(fetch_cdc_docs(CDC_GUIDELINE_URLS))
if KB_ENABLE_PUBMED:
    print(f"Fetching PubMed docs for query: {PUBMED_QUERY[:80]}...")
    try:
        CLINICAL_DOCS.extend(fetch_pubmed_docs(PUBMED_QUERY, retmax=PUBMED_RETMAX))
    except Exception as e:
        print(f"PubMed fetch failed: {e}")
if KB_ENABLE_OPENFDA:
    print(f"Fetching openFDA API drug safety docs for: {OPENFDA_TERMS}")
    CLINICAL_DOCS.extend(fetch_openfda_docs(OPENFDA_TERMS, per_term_limit=1))
if KB_ENABLE_OPENFDA_EVENT:
    print(f"Fetching openFDA adverse event docs for: {OPENFDA_EVENT_TERMS}")
    CLINICAL_DOCS.extend(fetch_openfda_adverse_event_docs(OPENFDA_EVENT_TERMS, per_term_limit=OPENFDA_EVENT_PER_TERM_LIMIT))
if KB_ENABLE_OPENFDA_ZIP:
    print(f"Loading FDA ZIP docs from: {OPENFDA_ZIP_DIR}")
    CLINICAL_DOCS.extend(fetch_openfda_zip_docs(OPENFDA_ZIP_DIR, max_docs=OPENFDA_ZIP_MAX_DOCS))

if not CLINICAL_DOCS:
    raise ValueError(
        "No knowledge docs loaded. Set KB_USE_SEED=true or enable KB_ENABLE_CDC / "
        "KB_ENABLE_PUBMED / KB_ENABLE_OPENFDA / KB_ENABLE_OPENFDA_EVENT / KB_ENABLE_OPENFDA_ZIP."
    )

# Chunk documents before indexing (RecursiveCharacterTextSplitter)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=RAG_CHUNK_SIZE,
    chunk_overlap=RAG_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
CHUNKED_DOCS = splitter.split_documents(CLINICAL_DOCS)

print(f"Building FAISS index from {len(CLINICAL_DOCS)} documents -> {len(CHUNKED_DOCS)} chunks...")
vectorstore = FAISS.from_documents(CHUNKED_DOCS, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 3})

source_counts: dict = {}
for d in CLINICAL_DOCS:
    src = d.metadata.get("category", "unknown")
    source_counts[src] = source_counts.get(src, 0) + 1
print(f"Vector store ready. Document categories: {source_counts}")

---
## Cell 6b — Optional: Hugging Face medical LLM checker
Use this to cross-check a clinical summary with a medical model served via Hugging Face Inference API.

Notes:
- Requires HF_API_TOKEN in .env
- Keep this as advisory only (not final diagnosis)

In [ ]:
import os
import re
import json
from huggingface_hub import InferenceClient
from huggingface_hub.utils import HfHubHTTPError

# You can optionally include provider routing in the model string, for example:
#   m42-health/Llama3-Med42-8B:featherless-ai
#   m42-health/Llama3-Med42-8B:cheapest
HF_MEDICAL_MODEL = os.getenv("HF_MEDICAL_MODEL", "m42-health/Llama3-Med42-8B")
HF_INFERENCE_PROVIDER = os.getenv("HF_INFERENCE_PROVIDER", "").strip()


def _normalize_hf_token(raw: str) -> str:
    """Normalize token values from .env or shell vars."""
    token = (raw or "").strip().strip('"').strip("'")
    if token.lower().startswith("bearer "):
        token = token.split(None, 1)[1].strip()
    return token


def _token_status(token: str) -> str:
    """Return a quick token status classification for user-facing diagnostics."""
    if not token:
        return "missing"
    placeholders = {"hf_...", "hf_xxx", "your_token_here", "your-hf-token"}
    if token in placeholders:
        return "placeholder"
    if not token.startswith("hf_"):
        return "malformed"
    if len(token) < 16:
        return "too_short"
    return "ok"


def _get_hf_token() -> tuple[str, str]:
    """Get HF token from common env vars and return (token, source_name)."""
    token = _normalize_hf_token(os.getenv("HF_API_TOKEN", ""))
    if token:
        return token, "HF_API_TOKEN"

    token = _normalize_hf_token(os.getenv("HF_TOKEN", ""))
    if token:
        return token, "HF_TOKEN"

    return "", "(none)"


def _candidate_models(base_model: str) -> list[str]:
    """Generate ordered candidate model strings for HF router/provider routing."""
    candidates = [base_model]

    # If no provider suffix is present, try configured provider,
    # then featherless-ai first, then cheapest as a fallback.
    if ":" not in base_model:
        if HF_INFERENCE_PROVIDER:
            candidates.append(f"{base_model}:{HF_INFERENCE_PROVIDER}")
        candidates.append(f"{base_model}:featherless-ai")
        candidates.append(f"{base_model}:cheapest")

    # Deduplicate while preserving order.
    seen = set()
    ordered = []
    for m in candidates:
        if m not in seen:
            seen.add(m)
            ordered.append(m)
    return ordered


def _is_model_not_supported_error(status: int | None, body_text: str) -> bool:
    """Detect HF router model/provider support errors."""
    if status != 400:
        return False
    txt = (body_text or "").lower()
    if "model_not_supported" in txt:
        return True
    if "not supported by any provider" in txt:
        return True
    return False


def medical_llm_check(clinical_prompt: str) -> str:
    """Advisory second opinion using a Hugging Face-hosted medical model."""
    token, token_source = _get_hf_token()
    token_state = _token_status(token)

    if token_state != "ok":
        return (
            "HF token is not ready (status: "
            f"{token_state}). Set HF_API_TOKEN or HF_TOKEN in .env, rerun Cell 2, then rerun this cell."
        )

    base_model = (os.getenv("HF_MEDICAL_MODEL", HF_MEDICAL_MODEL) or HF_MEDICAL_MODEL).strip()
    if not base_model:
        return "HF_MEDICAL_MODEL is empty. Set it in .env and rerun this cell."

    client = InferenceClient(api_key=token)
    messages = [
        {
            "role": "system",
            "content": (
                "You are an evidence-based clinical assistant. "
                "Return concise differential diagnosis, red flags, and safe next-step investigations. "
                "Do not provide definitive diagnosis."
            ),
        },
        {"role": "user", "content": clinical_prompt},
    ]

    attempted_models: list[str] = []
    last_error_text = ""

    for model in _candidate_models(base_model):
        attempted_models.append(model)
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=500,
                temperature=0.1,
            )
            content = response.choices[0].message.content
            if content:
                if model != base_model:
                    return f"[HF fallback used: {model}]\n\n{content}"
                return content
            return "HF response was empty."

        except HfHubHTTPError as e:
            status = getattr(getattr(e, "response", None), "status_code", None)
            body = getattr(getattr(e, "response", None), "text", "")
            body_preview = (body or str(e))[:500]
            last_error_text = body_preview

            # Try next routed model candidate for model/provider support mismatch.
            if _is_model_not_supported_error(status, body):
                continue

            if status == 401:
                return (
                    "HF authentication failed (401 Invalid username or password). "
                    "Your HF token is missing/invalid/revoked, or not passed correctly. "
                    f"Token source: {token_source}. Model: {model}."
                )
            if status == 403:
                return (
                    "HF access denied (403). Token is valid but this model/provider may be gated "
                    "or your account lacks permission. "
                    f"Model: {model}."
                )
            if status == 404:
                return (
                    "HF model endpoint not found (404). Check HF_MEDICAL_MODEL value "
                    f"('{model}')."
                )
            if status == 429:
                return "HF rate limit reached (429). Wait and retry."
            if status and status >= 500:
                return f"HF server error ({status}). Try again shortly."

            return f"HF API error ({status}): {body_preview}"

        except Exception as e:
            return f"Unexpected HF client error: {type(e).__name__}: {e}"

    return (
        "HF model routing failed: none of the candidate model routes are enabled for your account. "
        f"Tried: {attempted_models}. "
        "Set HF_MEDICAL_MODEL to a provider-routed model, e.g. "
        "m42-health/Llama3-Med42-8B:featherless-ai or m42-health/Llama3-Med42-8B:cheapest, "
        "or enable that provider in Hugging Face settings. "
        f"Last error: {last_error_text}"
    )


def medical_llm_diagnostics() -> None:
    """Lightweight diagnostics before invoking the medical checker."""
    token, token_source = _get_hf_token()
    token_state = _token_status(token)
    model = (os.getenv("HF_MEDICAL_MODEL", HF_MEDICAL_MODEL) or HF_MEDICAL_MODEL).strip()
    masked = f"{token[:6]}...{token[-4:]}" if len(token) > 12 else "(not available)"

    print("HF diagnostics:")
    print(f"  token_status : {token_state}")
    print(f"  token_source : {token_source}")
    print(f"  token_masked : {masked}")
    print(f"  model        : {model}")
    print(f"  provider_env : {HF_INFERENCE_PROVIDER or '(not set)'}")
    print(f"  model_tryset : {_candidate_models(model)}")


# ── RAG quality helpers (shared with clinical_node fallback logic) ─────────────

def _tokenize_words(text: str) -> set:
    return {w for w in re.findall(r"[a-zA-Z]{3,}", (text or "").lower())}


def _rag_miss_signal(query: str, docs: list) -> tuple[bool, float]:
    """Heuristic: low lexical overlap between query and top FAISS-retrieved docs."""
    q = _tokenize_words(query)
    if not q or not docs:
        return True, 0.0

    overlaps = []
    for d in docs:
        d_words = _tokenize_words(getattr(d, "page_content", ""))
        overlaps.append(len(q & d_words) / max(1, len(q)) if d_words else 0.0)

    best = max(overlaps) if overlaps else 0.0
    return best < 0.08, best


# Error-detection: returns True when medical_llm_check() returned an error string
_MED42_ERROR_PREFIXES = (
    "HF token", "HF_MEDICAL_MODEL", "HF response was empty",
    "HF authentication", "HF access denied", "HF model endpoint",
    "HF rate limit", "HF server error", "HF API error",
    "Unexpected HF", "HF model routing",
)

def _is_med42_error(text: str) -> bool:
    return any(text.startswith(p) for p in _MED42_ERROR_PREFIXES)


# Example quick check
example_prompt = (
    "58-year-old with exertional chest tightness, dyspnea, HTN, diabetes, on lisinopril/metformin, "
    "recent ibuprofen use, BP 158/96. Provide red flags and next investigations."
)

medical_llm_diagnostics()
print(medical_llm_check(example_prompt))

---
## Cell 7 — AuraState Definition
The shared typed state that flows through every node.

In [ ]:
from typing import TypedDict, Annotated, List, Optional
import operator

class AuraState(TypedDict):
    # ── Input ────────────────────────────────────────────────────────────────
    raw_transcript:       str
    session_id:           str
    patient_context:      dict

    # ── Optional STT metadata (pre-intake) ──────────────────────────────────
    stt_enabled:          bool
    transcript_source:    str

    # ── After intake ─────────────────────────────────────────────────────────
    scrubbed_transcript:  str
    pii_detected:         List[dict]

    # ── Supervisor routing ───────────────────────────────────────────────────
    agents_needed:        List[str]

    # ── Agent outputs (operator.add = append, not overwrite) ─────────────────
    clinical_findings:    Annotated[List[str], operator.add]
    drug_interactions:    Annotated[List[str], operator.add]
    research_notes:       Annotated[List[str], operator.add]

    # ── Final clinical output ────────────────────────────────────────────────
    soap_note:            str

    # ── IMDA Governance fields (Principles 1–5) ───────────────────────────────
    xai_record:           dict          # P2: Explainability record
    oversight_level:      str           # P1: advisory / mandatory / escalate
    oversight_instructions: str         # P1: human-readable action instructions
    human_review_required: bool         # P1: flag for doctor
    escalation_required:  bool          # P1: flag for immediate senior review
    fairness_passed:      bool          # P3: no biased language detected
    fairness_issues:      List[dict]    # P3: list of flagged issues
    pdpa_compliant:       bool          # P4: PII scrubbed
    output_blocked:       bool          # P6: blocked if below confidence threshold
    block_reason:         str           # P6: why output was blocked
    moh_compliant:        bool          # MOH: SaMD Class B disclosure attached
    samd_class:           str           # HSA: SaMD classification
    audit_log_path:       str           # P5: path to immutable audit log
    consultation_complete: bool

print("AuraState schema (with IMDA governance fields):")
for field, annotation in AuraState.__annotations__.items():
    marker = " [GOVERNANCE]" if field in (
        "xai_record","oversight_level","oversight_instructions",
        "human_review_required","escalation_required","fairness_passed",
        "fairness_issues","pdpa_compliant","output_blocked","block_reason",
        "moh_compliant","samd_class","audit_log_path"
    ) else ""
    print(f"  {field:<28} {str(annotation)[:40]}{marker}")


---
## Cell 8 — Agent Node Functions

In [ ]:
import uuid

# ── Helper: print what each node is doing ─────────────────────────────────────
def log(node: str, msg: str):
    print(f"  [{node.upper()}] {msg}")


# ── Node 0: STT prep — annotate transcript source before intake ───────────────
def stt_prep_node(state: AuraState) -> dict:
    source = state.get("transcript_source", "manual")
    stt_enabled = bool(state.get("stt_enabled", False))
    transcript = (state.get("raw_transcript", "") or "").strip()

    if not transcript:
        log("stt_prep", "No transcript provided; using placeholder text.")
        transcript = "No transcript provided."

    log("stt_prep", f"Transcript source: {source}")
    return {
        "raw_transcript": transcript,
        "transcript_source": source,
        "stt_enabled": stt_enabled,
    }


# ── Node 1: Intake — PII scrub ─────────────────────────────────────────────────
def intake_node(state: AuraState) -> dict:
    log("intake", "Scrubbing PII from transcript...")
    scrubbed, detected = scrub_pii(state["raw_transcript"])
    log("intake", f"Scrubbed {len(detected)} PII entities")
    return {
        "scrubbed_transcript": scrubbed,
        "pii_detected": detected,
    }


# ── Node 2: Supervisor — decides which specialist agents are needed ─────────────
def supervisor_node(state: AuraState) -> dict:
    log("supervisor", "Analysing transcript to determine required agents...")

    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content="""You are a medical consultation router.
Analyse the transcript and return a JSON object with key 'agents' containing a list
of required specialist agents from: [\"clinical\", \"drug\", \"research\"].
- clinical: always include for any consultation
- drug: include if any medications are mentioned
- research: include if rare conditions, unclear diagnosis, or recent research needed
Return ONLY valid JSON, no explanation. Example: {\"agents\": [\"clinical\", \"drug\"]}"""),
        HumanMessage(content=f"Transcript:\n{state['scrubbed_transcript']}")
    ])

    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({})

    try:
        # Strip markdown code fences if present
        clean = result.strip().strip("```json").strip("```").strip()
        agents = json.loads(clean).get("agents", ["clinical"])
    except Exception:
        agents = ["clinical"]  # safe fallback

    log("supervisor", f"Routing to agents: {agents}")
    return {"agents_needed": agents}


# Ensure shared helper functions exist even if helper cells were not run first.
if "_rag_miss_signal" not in globals():
    def _tokenize_words(text: str) -> set[str]:
        return set(re.findall(r"[a-z0-9]+", (text or "").lower()))

    def _rag_miss_signal(query: str, docs: list) -> tuple[bool, float]:
        query_words = _tokenize_words(query)
        if not query_words or not docs:
            return True, 0.0

        best_overlap = 0.0
        for doc in docs:
            doc_words = _tokenize_words(getattr(doc, "page_content", ""))
            if not doc_words:
                continue
            overlap = len(query_words & doc_words) / max(len(query_words), 1)
            best_overlap = max(best_overlap, overlap)

        return best_overlap < 0.08, best_overlap

if "_is_med42_error" not in globals():
    def _is_med42_error(text: str) -> bool:
        value = (text or "").lower()
        return any(marker in value for marker in [
            "hf token is not ready",
            "authentication failed",
            "access denied",
            "not supported by any provider",
            "response was empty",
        ])


# ── Node 3: Clinical Agent — FAISS RAG → Med42 fallback → Claude last resort ──
def clinical_node(state: AuraState) -> dict:
    log("clinical", "Retrieving clinical guidelines from FAISS...")

    docs = retriever.invoke(state["scrubbed_transcript"])
    context = "\n".join([f"[{d.metadata['source']}] {d.page_content}" for d in docs])
    is_miss, overlap = _rag_miss_signal(state["scrubbed_transcript"], docs)

    # ── Tier 1 complete. If RAG quality is adequate, go straight to Claude. ──
    if not is_miss:
        log("clinical", f"FAISS RAG adequate (overlap={overlap:.3f}) — using Claude Haiku")
        prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content="""You are a clinical decision support AI.
Using the provided guideline context, analyse the consultation transcript.
Provide: 1) Key clinical findings  2) Differential diagnoses  3) Recommended investigations
Be concise and clinically precise."""),
            HumanMessage(content=f"Transcript:\n{state['scrubbed_transcript']}\n\nGuideline context:\n{context}")
        ])
        chain = prompt | llm | StrOutputParser()
        finding = chain.invoke({})
        log("clinical", f"Finding generated ({len(finding)} chars)")
        return {"clinical_findings": [f"[Source: Claude Haiku + FAISS RAG]\n\n{finding}"]}

    # ── Tier 2: RAG miss — try Med42 (HuggingFace) ────────────────────────────
    log("clinical", f"FAISS RAG miss (overlap={overlap:.3f}) — escalating to Med42...")
    fallback_prompt = (
        "Provide differential diagnoses, critical red flags, and urgent next-step investigations "
        "based on this de-identified clinical transcript:\n\n"
        + state["scrubbed_transcript"]
    )
    med42_result = medical_llm_check(fallback_prompt)

    if not _is_med42_error(med42_result):
        log("clinical", "Med42 advisory used as primary clinical finding")
        finding = f"[Source: Med42 advisory — FAISS RAG miss, overlap={overlap:.3f}]\n\n{med42_result}"
        return {"clinical_findings": [finding]}

    # ── Tier 3: Med42 unavailable — fall back to Claude without RAG context ──
    log("clinical", f"Med42 unavailable ({med42_result[:60]}…) — last resort: Claude Haiku")
    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content="""You are a clinical decision support AI.
The knowledge base did not return relevant guidelines for this consultation.
Analyse the transcript using your clinical knowledge.
Provide: 1) Key clinical findings  2) Differential diagnoses  3) Recommended investigations
Be concise and clinically precise."""),
        HumanMessage(content=f"Transcript:\n{state['scrubbed_transcript']}")
    ])
    chain = prompt | llm | StrOutputParser()
    finding = chain.invoke({})
    log("clinical", f"Finding generated ({len(finding)} chars)")
    return {"clinical_findings": [
        f"[Source: Claude Haiku (last resort — FAISS RAG miss overlap={overlap:.3f}, Med42 unavailable)]\n\n{finding}"
    ]}


# ── Node 4: Drug Agent — interaction checker ───────────────────────────────────
def drug_node(state: AuraState) -> dict:
    log("drug", "Checking drug interactions...")

    docs = retriever.invoke("drug interaction " + state["scrubbed_transcript"])
    context = "\n".join([f"[{d.metadata['source']}] {d.page_content}" for d in docs])

    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content="""You are a clinical pharmacist AI.
Identify all medications mentioned, check for interactions, and flag contraindications.
Format: list each drug, its indication, key interactions, and monitoring requirements."""),
        HumanMessage(content=f"Transcript:\n{state['scrubbed_transcript']}\n\nDrug database context:\n{context}")
    ])

    chain = prompt | llm | StrOutputParser()
    interaction = chain.invoke({})
    log("drug", f"Drug review generated ({len(interaction)} chars)")
    return {"drug_interactions": [interaction]}


# ── Node 5: Research Agent — evidence synthesis ────────────────────────────────
def research_node(state: AuraState) -> dict:
    log("research", "Synthesising research context...")

    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content="""You are a medical research AI.
Based on the clinical transcript, provide relevant evidence-based context:
recent guideline updates, epidemiology, and any emerging treatment options.
Keep it clinically actionable."""),
        HumanMessage(content=f"Transcript:\n{state['scrubbed_transcript']}")
    ])

    chain = prompt | llm | StrOutputParser()
    research = chain.invoke({})
    log("research", f"Research note generated ({len(research)} chars)")
    return {"research_notes": [research]}


# ── Node 6: Summary — synthesise SOAP note ─────────────────────────────────────
def summary_node(state: AuraState) -> dict:
    log("summary", "Synthesising final SOAP note...")

    all_findings = []
    if state.get("clinical_findings"):
        all_findings.append("=== CLINICAL FINDINGS ===\n" + "\n".join(state["clinical_findings"]))
    if state.get("drug_interactions"):
        all_findings.append("=== DRUG REVIEW ===\n" + "\n".join(state["drug_interactions"]))
    if state.get("research_notes"):
        all_findings.append("=== RESEARCH CONTEXT ===\n" + "\n".join(state["research_notes"]))

    combined = "\n\n".join(all_findings)
    patient = state.get("patient_context", {})

    prompt = ChatPromptTemplate.from_messages([
        SystemMessage(content="""You are a senior clinician writing a SOAP note.
Synthesise the agent findings into a structured SOAP note:

SUBJECTIVE: Patient complaints and history from transcript
OBJECTIVE: Relevant vitals, examination findings, and lab values mentioned
ASSESSMENT: Primary diagnosis and differentials with clinical reasoning
PLAN: Medications, investigations, referrals, and follow-up schedule

Be precise, use medical terminology, keep it clinically actionable."""),
        HumanMessage(content=f"Patient context: {json.dumps(patient)}\n\nAgent findings:\n{combined}\n\nOriginal transcript (de-identified):\n{state['scrubbed_transcript']}")
    ])

    chain = prompt | llm | StrOutputParser()
    soap = chain.invoke({})
    log("summary", f"SOAP note generated ({len(soap)} chars)")
    return {
        "soap_note": soap,
        "consultation_complete": True
    }


print("All 7 agent nodes defined:")
for n in ["stt_prep", "intake", "supervisor", "clinical", "drug", "research", "summary"]:
    print(f"  {n}_node")
print()
print("clinical_node fallback chain: FAISS RAG → Med42 (HF) → Claude Haiku (last resort)")

---
## Cell 9 — Routing Logic & Graph Assembly

---
## Cell 9b — Install governance dependencies
Required for IMDA AI Governance layers (XAI, fairness, audit, MOH safety guard).

In [ ]:
import subprocess, sys
gov_packages = ["pytest", "pytest-mock", "bandit", "safety", "nbstripout"]
for p in gov_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=True)
print("Governance dependencies installed.")

---
## Cell 9c — Load IMDA Governance modules
Imports all six governance nodes aligned with the IMDA Model AI Governance Framework 2nd Edition.

```
governance/
  xai_layer.py            # Principle 2 — Explainability
  human_oversight.py      # Principle 1 — Human oversight
  fairness_monitor.py     # Principle 3 — Fairness
  audit_log.py            # Principle 5 — Accountability
  clinical_safety_guard.py  # MOH / HSA SaMD Class B
```

In [ ]:
import sys, os

# Add project root to path so governance modules resolve
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

try:
    from governance.xai_layer            import xai_node
    from governance.human_oversight      import human_oversight_node
    from governance.fairness_monitor     import fairness_node
    from governance.audit_log            import audit_node
    from governance.clinical_safety_guard import clinical_safety_guard_node
    GOVERNANCE_ENABLED = True
    print("IMDA governance modules loaded successfully.")
    print("  xai_node              — Principle 2: Explainability")
    print("  human_oversight_node  — Principle 1: Human oversight")
    print("  fairness_node         — Principle 3: Fairness")
    print("  clinical_safety_guard — MOH/HSA SaMD Class B")
    print("  audit_node            — Principle 5: Accountability")
except ImportError as e:
    print(f"WARNING: Governance modules not found — {e}")
    print("Make sure you are running from the aura-health/ project directory.")
    GOVERNANCE_ENABLED = False

    # Stub fallback nodes so graph still compiles without governance
    def xai_node(state):              return {}
    def human_oversight_node(state):  return {"oversight_level": "advisory", "oversight_instructions": "Review required", "human_review_required": True, "escalation_required": False}
    def fairness_node(state):         return {"fairness_passed": True, "fairness_issues": [], "pdpa_compliant": True}
    def clinical_safety_guard_node(state): return {"output_blocked": False, "moh_compliant": True, "samd_class": "Class B CDS"}
    def audit_node(state):            return {"audit_log_path": None, "consultation_complete": True}

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# ── Routing function (unchanged) ─────────────────────────────────────────────
def route_to_agents(state: AuraState) -> List[str]:
    needed = state.get("agents_needed", ["clinical"])
    routes = []
    if "clinical"  in needed: routes.append("clinical")
    if "drug"      in needed: routes.append("drug")
    if "research"  in needed: routes.append("research")
    return routes if routes else ["clinical"]

# ── Build the StateGraph ──────────────────────────────────────────────────────
workflow = StateGraph(AuraState)

# STT + clinical pipeline nodes
workflow.add_node("stt_prep",   stt_prep_node)
workflow.add_node("intake",     intake_node)
workflow.add_node("supervisor", supervisor_node)
workflow.add_node("clinical",   clinical_node)
workflow.add_node("drug",       drug_node)
workflow.add_node("research",   research_node)
workflow.add_node("summary",    summary_node)

# IMDA Governance layer nodes
workflow.add_node("xai",             xai_node)              # P2 Explainability
workflow.add_node("fairness",        fairness_node)         # P3 Fairness
workflow.add_node("human_oversight", human_oversight_node)  # P1 Human oversight
workflow.add_node("clinical_safety", clinical_safety_guard_node)  # MOH/HSA
workflow.add_node("audit",           audit_node)            # P5 Accountability

# ── Clinical pipeline edges ───────────────────────────────────────────────────
workflow.set_entry_point("stt_prep")
workflow.add_edge("stt_prep", "intake")
workflow.add_edge("intake", "supervisor")
workflow.add_conditional_edges(
    "supervisor",
    route_to_agents,
    {"clinical": "clinical", "drug": "drug", "research": "research"}
)
workflow.add_edge("clinical",  "summary")
workflow.add_edge("drug",      "summary")
workflow.add_edge("research",  "summary")

# ── IMDA Governance pipeline (sequential after summary) ───────────────────────
workflow.add_edge("summary",          "xai")             # attach explainability
workflow.add_edge("xai",              "fairness")        # check for bias
workflow.add_edge("fairness",         "human_oversight") # classify oversight level
workflow.add_edge("human_oversight",  "clinical_safety") # apply MOH thresholds
workflow.add_edge("clinical_safety",  "audit")           # write immutable log
workflow.add_edge("audit",            END)

checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

print("Graph compiled with optional STT entry and IMDA governance layer.")
print("")
print("Clinical pipeline: stt_prep → intake → supervisor → [clinical|drug|research] → summary")
print("Governance layer:  summary → xai → fairness → human_oversight → clinical_safety → audit → END")


---
## Cell 10 — Visualise the Graph Topology
Renders the actual compiled graph as a diagram inline.

In [ ]:
from IPython.display import Image, display

try:
    img = graph.get_graph().draw_mermaid_png()
    display(Image(img))
    print("Graph topology rendered above.")
except Exception as e:
    # Fallback: print Mermaid source
    print("Could not render PNG (install playwright for images).")
    print("Mermaid source:")
    print(graph.get_graph().draw_mermaid())

---
## Cell 10a — Optional STT (Mic -> OpenAI API)
Set `USE_STT_MIC = True` to record from your microphone in chunks and transcribe using the OpenAI API.
Default STT model is `gpt-4o-mini` (with automatic fallback to transcription-capable models if needed).
If disabled or transcription is empty, Cell 11 falls back to the manual `TRANSCRIPT` text.

In [ ]:
# Optional STT (microphone -> chunked audio -> OpenAI API transcription)
# Default behavior can be controlled via .env: USE_STT_MIC_DEFAULT=true|false

import os

def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return str(raw).strip().lower() in {"1", "true", "yes", "on"}

USE_STT_MIC = _env_bool("USE_STT_MIC_DEFAULT", False)
STT_OPENAI_MODEL = os.getenv("STT_OPENAI_MODEL", "gpt-4o-mini")
STT_OPENAI_FALLBACK_MODELS = [
    m.strip() for m in os.getenv("STT_OPENAI_FALLBACK_MODELS", "gpt-4o-mini-transcribe,whisper-1").split(",")
    if m.strip() and m.strip() != STT_OPENAI_MODEL
]
STT_LANGUAGE = os.getenv("STT_LANGUAGE", "en")
STT_SAMPLE_RATE = int(os.getenv("STT_SAMPLE_RATE", "16000"))
STT_CHUNK_SECONDS = int(os.getenv("STT_CHUNK_SECONDS", "8"))
STT_NUM_CHUNKS = int(os.getenv("STT_NUM_CHUNKS", "3"))
STT_ENABLE_VOICE_STOP = _env_bool("STT_ENABLE_VOICE_STOP", True)
STT_VOICE_STOP_KEYWORDS = [
    k.strip().lower()
    for k in os.getenv("STT_VOICE_STOP_KEYWORDS", "stop recording,end transcript,stop").split(",")
    if k.strip()
]

STT_TRANSCRIPT = ""

if USE_STT_MIC:
    import importlib
    import subprocess
    import sys
    import tempfile

    # Lazy-install only when STT is enabled.
    optional_deps = {
        "openai": "openai",
        "sounddevice": "sounddevice",
        "soundfile": "soundfile",
    }
    for dep, module_name in optional_deps.items():
        try:
            importlib.import_module(module_name)
        except Exception:
            print(f"Installing optional STT dependency: {dep}")
            subprocess.run([sys.executable, "-m", "pip", "install", dep, "-q"], check=True)

    import sounddevice as sd
    import soundfile as sf
    from openai import OpenAI

    api_key = os.getenv("OPENAI_API_KEY", "").strip()
    if not api_key:
        print("OPENAI_API_KEY not set. Falling back to manual transcript in Cell 11.")
    else:
        client = OpenAI(api_key=api_key)

        print("=" * 72)
        print("Optional STT is ON — recording from microphone in chunks")
        print("=" * 72)
        print(
            f"Primary model: {STT_OPENAI_MODEL} | "
            f"Chunks: {STT_NUM_CHUNKS} x {STT_CHUNK_SECONDS}s | Language: {STT_LANGUAGE}"
        )
        if STT_ENABLE_VOICE_STOP:
            print(f"Voice-stop keywords: {STT_VOICE_STOP_KEYWORDS}")
            print("Say one of the keywords to stop recording early.")
        print("Tip: Use Jupyter Interrupt to stop immediately and keep captured chunks.")

        chunk_texts = []

        try:
            for i in range(STT_NUM_CHUNKS):
                print(f"\nRecording chunk {i + 1}/{STT_NUM_CHUNKS}...")
                audio = sd.rec(
                    int(STT_CHUNK_SECONDS * STT_SAMPLE_RATE),
                    samplerate=STT_SAMPLE_RATE,
                    channels=1,
                    dtype="float32",
                )
                sd.wait()

                wav_path = None
                try:
                    with tempfile.NamedTemporaryFile(suffix=f"_chunk{i+1}.wav", delete=False) as tmp:
                        wav_path = tmp.name
                    sf.write(wav_path, audio, STT_SAMPLE_RATE)

                    attempted_models = [STT_OPENAI_MODEL] + STT_OPENAI_FALLBACK_MODELS
                    text = ""
                    last_err = ""

                    for m in attempted_models:
                        try:
                            with open(wav_path, "rb") as audio_file:
                                out = client.audio.transcriptions.create(
                                    model=m,
                                    file=audio_file,
                                    language=STT_LANGUAGE,
                                )
                            text = (getattr(out, "text", "") or "").strip()
                            if m != STT_OPENAI_MODEL:
                                print(f"  STT fallback model used: {m}")
                            break
                        except Exception as e:
                            last_err = f"{type(e).__name__}: {e}"

                    if not text and last_err:
                        print(f"  STT chunk failed after model fallbacks: {last_err}")

                    chunk_texts.append(text)
                    print(f"Chunk {i + 1} transcript: {text[:160]}{'...' if len(text) > 160 else ''}")

                    lower_text = text.lower()
                    if STT_ENABLE_VOICE_STOP and any(k in lower_text for k in STT_VOICE_STOP_KEYWORDS):
                        print("Voice-stop keyword detected. Ending STT recording loop early.")
                        break
                finally:
                    if wav_path and os.path.exists(wav_path):
                        os.remove(wav_path)
        except KeyboardInterrupt:
            print("\nRecording interrupted by user. Finalizing transcript from captured chunks...")

        STT_TRANSCRIPT = " ".join([t for t in chunk_texts if t]).strip()
        if STT_TRANSCRIPT:
            print("\nFinal STT transcript captured successfully.")
            print(f"Transcript preview: {STT_TRANSCRIPT[:240]}{'...' if len(STT_TRANSCRIPT) > 240 else ''}")
        else:
            print("\nNo speech recognized from microphone chunks; fallback to manual transcript will be used.")
else:
    print("Optional STT is OFF. Cell 11 will use the manual TRANSCRIPT variable.")
    print("Tip: set USE_STT_MIC_DEFAULT=true in .env to auto-enable mic STT.")

---
## Cell 10b — Optional STT smoke test (no mic)
Use this to simulate STT output quickly without audio recording.
Set `STT_SMOKE_TEST = True` to inject sample transcript text into `STT_TRANSCRIPT`.
You can also inject `STT_PATIENT_CONTEXT`, which Cell 11 will use automatically when STT mode is active.

In [ ]:
# Optional STT smoke test (no microphone)
import os

def _env_bool_local(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return str(raw).strip().lower() in {"1", "true", "yes", "on"}

STT_SMOKE_TEST = _env_bool_local("STT_SMOKE_TEST", False)

STT_SMOKE_TEXT = """
Doctor: Please summarize this consult.
Patient: I have shortness of breath and chest tightness for 2 days with exertion.
Nurse: BP 160 over 110, pulse 120, oxygen saturation 96 percent on room air.
Doctor: The patient has hypertension currently on lorsartan potassium tablet 50mg & amLODIPINE tablet 5mg.
There is a unknown bump / swelling on the left leg that started 3 days ago. Minor pain whole day. Not sure if the swelling got to do with high urine acid level. 
I have been taking ibuprofen for the pain but it doesn't seem to help much. I am worried if this could be something serious.
Provide red flags and next investigations.

""".strip()

# Optional: when STT smoke mode is enabled, this context can override Cell 11 PATIENT_CONTEXT.
STT_SMOKE_PATIENT_CONTEXT = {
    "age": 34,
    "gender": "male",
    "known_conditions": ["hypertension"],
    "current_medications": ["lorsartan potassium 50mg", "amlodipine 5mg"],
    "allergies": "Amoxilin (rash)",
}

# Reset defaults to avoid stale values across reruns.
STT_PATIENT_CONTEXT = None

if STT_SMOKE_TEST:
    STT_TRANSCRIPT = STT_SMOKE_TEXT
    STT_PATIENT_CONTEXT = STT_SMOKE_PATIENT_CONTEXT
    USE_STT_MIC = True
    print("STT smoke test ON: injected sample transcript into STT_TRANSCRIPT.")
    print("STT smoke test ON: injected STT_PATIENT_CONTEXT for Cell 11.")
    print(f"Preview: {STT_TRANSCRIPT[:220]}{'...' if len(STT_TRANSCRIPT) > 220 else ''}")
else:
    print("STT smoke test OFF (set STT_SMOKE_TEST=true in .env to enable by default).")

---
## Cell 11 — Run a Consultation
Edit `TRANSCRIPT` and `PATIENT_CONTEXT`, or run Cell 10a/10b first to use optional STT transcript input.

In [ ]:
import uuid
from datetime import datetime

# ── Manual fallback transcript (used when STT is disabled/unavailable) ─────────
TRANSCRIPT = """
Doctor: Good morning. What brings you in today?

Patient: I've been having chest tightness for the past 3 days, especially when I climb stairs.
I also feel short of breath. My name is Michael Tan, I'm 58 years old.

Doctor: Any history of heart problems?

Patient: I have hypertension, diagnosed 2 years ago. I'm currently on lisinopril 10mg daily.
I also have Type 2 diabetes, taking metformin 500mg twice a day. My last HbA1c was 7.8%.

Doctor: What are your vitals today?

Nurse: BP 158/96 mmHg, HR 88 bpm, SpO2 96%, Temp 37.1C, RR 18.

Doctor: Any chest pain at rest, palpitations, or leg swelling?

Patient: No chest pain at rest. Some mild ankle swelling on the right side.
I've also been taking ibuprofen for back pain this week, about 400mg twice daily.

Doctor: Any allergies?

Patient: No known drug allergies. Contact: +65-9234-5678, email: michael.tan@email.com
"""

DEFAULT_PATIENT_CONTEXT = {
    "age": 58,
    "gender": "male",
    "known_conditions": ["hypertension", "type_2_diabetes"],
    "current_medications": ["lisinopril 10mg", "metformin 500mg BD"],
    "allergies": "NKDA"
}

# Optional override for UI/API-style injection when running inside the notebook.
FRONTEND_PATIENT_CONTEXT = globals().get("FRONTEND_PATIENT_CONTEXT", None)

def normalize_patient_context(override=None, base=None):
    context = dict(base or DEFAULT_PATIENT_CONTEXT)
    override = override or {}

    if not isinstance(override, dict):
        return context

    for key in ["age", "gender", "allergies"]:
        value = override.get(key)
        if value not in (None, ""):
            context[key] = value

    for key in ["known_conditions", "current_medications"]:
        value = override.get(key)
        if value is None:
            continue
        if isinstance(value, str):
            context[key] = [item.strip() for item in value.split(",") if item.strip()]
        elif isinstance(value, list):
            context[key] = [str(item).strip() for item in value if str(item).strip()]

    return context

PATIENT_CONTEXT = normalize_patient_context(FRONTEND_PATIENT_CONTEXT)

# ── Choose transcript source and optional STT patient context ─────────────────
use_stt_mic = bool(globals().get("USE_STT_MIC", False))
stt_transcript = (globals().get("STT_TRANSCRIPT", "") or "").strip()
stt_patient_context = globals().get("STT_PATIENT_CONTEXT", None)

if use_stt_mic and stt_transcript:
    transcript_for_run = stt_transcript
    transcript_source = "stt_openai_mic"

    if isinstance(stt_patient_context, dict) and stt_patient_context:
        patient_context_for_run = normalize_patient_context(stt_patient_context)
        patient_context_source = "stt_injected"
    else:
        patient_context_for_run = PATIENT_CONTEXT
        patient_context_source = "manual_default"

elif use_stt_mic and not stt_transcript:
    transcript_for_run = TRANSCRIPT
    transcript_source = "manual_fallback_after_stt"
    patient_context_for_run = PATIENT_CONTEXT
    patient_context_source = "manual_default"
    print("STT was enabled but transcript is empty. Falling back to manual TRANSCRIPT.")

else:
    transcript_for_run = TRANSCRIPT
    transcript_source = "manual"
    patient_context_for_run = PATIENT_CONTEXT
    patient_context_source = "manual_default"

# ── Run the graph ──────────────────────────────────────────────────────────────
SESSION_ID = f"aura-{uuid.uuid4().hex[:8]}"
config = {"configurable": {"thread_id": SESSION_ID}}

print("=" * 60)
print("Aura Health Consultation")
print(f"Session ID         : {SESSION_ID}")
print(f"Transcript source  : {transcript_source}")
print(f"Patient context    : {patient_context_source}")
print(f"Effective context  : {patient_context_for_run}")
print(f"Started at         : {datetime.now().strftime('%H:%M:%S')}")
print("=" * 60)
print()

initial_state = {
    "raw_transcript":    transcript_for_run,
    "session_id":        SESSION_ID,
    "patient_context":   patient_context_for_run,
    "stt_enabled":       use_stt_mic,
    "transcript_source": transcript_source,
    "clinical_findings": [],
    "drug_interactions": [],
    "research_notes":    [],
}

# graph.invoke() runs synchronously — works perfectly in Jupyter
final_state = graph.invoke(initial_state, config=config)

print()
print(f"Completed at       : {datetime.now().strftime('%H:%M:%S')}")
print(f"Agents run         : {final_state.get('agents_needed', [])}")
print(f"PII scrubbed       : {len(final_state.get('pii_detected', []))} entities")

---
## Cell 12 — Display Results

In [ ]:
from IPython.display import Markdown, display

def print_section(title: str, content: str, color: str = "#0f6e56"):
    display(Markdown(f"### {title}"))
    display(Markdown(content))
    display(Markdown("---"))

display(Markdown("# Aura Health — Consultation Results"))
display(Markdown(f"**Session:** `{SESSION_ID}`"))

# PII Report
pii = final_state.get("pii_detected", [])
print_section(
    f"PII Scrub Report ({len(pii)} entities removed)",
    "\n".join([f"- `{p['type']}` (confidence: {p['score']})" for p in pii]) or "_None detected_"
)

# De-identified transcript
print_section(
    "De-identified Transcript",
    f"```\n{final_state.get('scrubbed_transcript', '')}\n```"
)

# Clinical findings
for i, finding in enumerate(final_state.get("clinical_findings", []), 1):
    print_section(f"Clinical Agent Finding #{i}", finding)

# Drug interactions
for i, drug in enumerate(final_state.get("drug_interactions", []), 1):
    print_section(f"Drug Agent Review #{i}", drug)

# Research notes
for i, research in enumerate(final_state.get("research_notes", []), 1):
    print_section(f"Research Agent Note #{i}", research)

# SOAP note — the main output
display(Markdown("## Final SOAP Note"))
display(Markdown(final_state.get("soap_note", "_Not generated_")))

---
## Cell 12a — Multi-scenario consultation demos
This section demonstrates two additional consultation patterns:
1. A likely RAG-miss case that falls back to Hugging Face Med42 (`medical_llm_check`).
2. A research-focused case using live PubMed retrieval (non-seeded corpus) with temporary retriever swap.

In [ ]:
import uuid

# _tokenize_words, _rag_miss_signal, and _is_med42_error are defined in Cell 6b.

print("=" * 72)
print("Scenario A: RAG miss -> clinical_node fallback chain (Med42 → Claude)")
print("=" * 72)

TRANSCRIPT_RAG_MISS = """
Doctor: Tell me what happened.
Patient: I did a technical deep-sea dive and now have joint pain, skin mottling,
and shortness of breath after surfacing. I am worried about decompression illness.
Doctor: Current vitals?
Nurse: BP 128/78, HR 110, SpO2 92%, RR 24, Temp 37.0C.
"""

PATIENT_RAG_MISS = {"age": 41, "gender": "male", "context": "post-dive symptoms"}

session_rag_miss = f"aura-{uuid.uuid4().hex[:8]}"
state_rag_miss = {
    "raw_transcript": TRANSCRIPT_RAG_MISS,
    "session_id": session_rag_miss,
    "patient_context": PATIENT_RAG_MISS,
    "clinical_findings": [],
    "drug_interactions": [],
    "research_notes": [],
}

# The graph now handles the full fallback chain inside clinical_node:
#   Tier 1 — FAISS RAG + Claude (adequate overlap)
#   Tier 2 — Med42 HF advisory (RAG miss)
#   Tier 3 — Claude last resort (RAG miss + Med42 unavailable)
result_rag_miss = graph.invoke(state_rag_miss, config={"configurable": {"thread_id": session_rag_miss}})

# Inspect which tier fired by reading the source tag embedded in the finding
findings = result_rag_miss.get("clinical_findings", [""])
finding_head = (findings[0] if findings else "")[:120]

retrieved_docs = retriever.invoke(result_rag_miss.get("scrubbed_transcript", TRANSCRIPT_RAG_MISS))
is_miss, overlap = _rag_miss_signal(result_rag_miss.get("scrubbed_transcript", TRANSCRIPT_RAG_MISS), retrieved_docs)

print(f"Session ID         : {session_rag_miss}")
print(f"Agents run         : {result_rag_miss.get('agents_needed', [])}")
print(f"RAG miss heuristic : {is_miss} (best_overlap={overlap:.3f})")
print(f"Clinical tier used : {finding_head}")

print("\nSOAP preview:")
print(result_rag_miss.get("soap_note", "")[:600] + "...")

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.messages import HumanMessage, SystemMessage

# _is_med42_error is defined in Cell 6b — used in the Tier 2 failure check below.

print("=" * 72)
print("Scenario B: Research-triggered consultation with live PubMed docs (non-seeded)")
print("=" * 72)

TRANSCRIPT_RESEARCH = """
Doctor: Please summarize this unusual neurology case.
Patient: Progressive proximal muscle weakness, heliotrope rash, Gottron papules,
and elevated CK. Steroid response was partial.
Doctor: We need latest evidence and emerging options beyond standard care.
"""

PATIENT_RESEARCH = {"age": 36, "gender": "female", "known_conditions": ["suspected dermatomyositis"]}
PUBMED_QUERY_DEMO = (
    "dermatomyositis adult treatment guideline randomized trial interstitial lung disease"
)

FALLBACK_PROMPT = (
    f"Patient: {PATIENT_RESEARCH}.\n\n"
    f"Consultation transcript:\n{TRANSCRIPT_RESEARCH}\n\n"
    "Provide an evidence-based clinical summary covering: differential diagnosis, "
    "current treatment guidelines, red flags to monitor, and next investigation steps. "
    "Focus on dermatomyositis with interstitial lung disease. Advisory only."
)

# ── Tier 1: Live PubMed ───────────────────────────────────────────────────────
print(f"\n[Tier 1] Fetching live PubMed docs for: {PUBMED_QUERY_DEMO}")
live_pubmed_docs = []
pubmed_error = None
try:
    live_pubmed_docs = fetch_pubmed_docs(PUBMED_QUERY_DEMO, retmax=8)
except Exception as e:
    pubmed_error = str(e)

if live_pubmed_docs:
    print(f"Live PubMed docs fetched: {len(live_pubmed_docs)}")
    temp_vs = FAISS.from_documents(live_pubmed_docs, embeddings)
    temp_retriever = temp_vs.as_retriever(search_kwargs={"k": 3})

    _original_retriever = retriever
    try:
        retriever = temp_retriever

        session_research = f"aura-{uuid.uuid4().hex[:8]}"
        state_research = {
            "raw_transcript": TRANSCRIPT_RESEARCH,
            "session_id": session_research,
            "patient_context": PATIENT_RESEARCH,
            "clinical_findings": [],
            "drug_interactions": [],
            "research_notes": [],
        }

        result_research = graph.invoke(
            state_research,
            config={"configurable": {"thread_id": session_research}},
        )

        print(f"Session ID     : {session_research}")
        print(f"Agents run     : {result_research.get('agents_needed', [])}")
        print("Top live sources used:")
        for d in temp_retriever.invoke(result_research.get("scrubbed_transcript", TRANSCRIPT_RESEARCH)):
            src = d.metadata.get("source", "unknown")
            print(f"  - {src}")

        if "research" in result_research.get("agents_needed", []):
            note = (result_research.get("research_notes", [""])[0] or "")[:1200]
            print("\nResearch note preview:")
            print(note + ("..." if len(note) >= 1200 else ""))
        else:
            print("\nResearch agent was not selected by supervisor in this run.")
            print("Tip: include explicit wording like 'latest evidence', 'recent trial', or 'emerging therapy'.")
    finally:
        retriever = _original_retriever
        print("\nRetriever restored to the default project knowledge base.")

else:
    # PubMed returned nothing (network/NCBI limit/quota issue)
    reason = pubmed_error or "0 abstracts returned (NCBI limit or network issue)"
    print(f"PubMed unavailable: {reason}")

    # ── Tier 2: Med42 via HuggingFace Inference API ───────────────────────────
    print("\n[Tier 2] Falling back to HuggingFace Med42 advisory...")
    med42_result = medical_llm_check(FALLBACK_PROMPT)

    if not _is_med42_error(med42_result):
        print("\n[Med42 advisory output]")
        print(med42_result[:2000] + ("..." if len(med42_result) > 2000 else ""))
    else:
        # ── Tier 3: AWS Bedrock (Claude) — last resort ────────────────────────
        print(f"Med42 unavailable: {med42_result[:200]}")
        print("\n[Tier 3] Falling back to AWS Bedrock (Claude) as last resort...")
        try:
            bedrock_messages = [
                SystemMessage(content=(
                    "You are an evidence-based clinical consultation assistant. "
                    "Provide a concise advisory summary: differential diagnosis, "
                    "treatment guidelines, red flags, and next investigations. "
                    "This is advisory only — not a definitive diagnosis."
                )),
                HumanMessage(content=FALLBACK_PROMPT),
            ]
            bedrock_response = llm.invoke(bedrock_messages)
            bedrock_text = (
                bedrock_response.content
                if hasattr(bedrock_response, "content")
                else str(bedrock_response)
            )
            print("\n[Bedrock (Claude) advisory output]")
            print(bedrock_text[:2000] + ("..." if len(bedrock_text) > 2000 else ""))
        except Exception as e:
            print(f"Bedrock fallback also failed: {type(e).__name__}: {e}")
            print(
                "\nAll three tiers exhausted (PubMed, Med42, Bedrock). "
                "Check network access, API keys, and AWS Bedrock model permissions."
            )

---
## Cell 12b — Display governance results
Shows XAI record, oversight level, fairness report, and audit log path.

In [ ]:
from IPython.display import Markdown, display

def display_governance_report(state: dict):
    """Display a formatted IMDA governance report for a completed consultation."""

    display(Markdown("## IMDA AI Governance Report"))
    display(Markdown(f"**Session:** `{state.get('session_id', 'unknown')}`"))

    # ── Oversight ─────────────────────────────────────────────────────────────
    level   = state.get("oversight_level", "unknown")
    blocked = state.get("output_blocked", False)
    colour  = {"advisory": "green", "mandatory": "orange",
               "escalate": "red", "autonomous": "blue"}.get(level, "grey")
    display(Markdown(
        f"### P1 — Human oversight\n"
        f"**Level:** {level.upper()}  \n"
        f"**Human review required:** {state.get('human_review_required', True)}  \n"
        f"**Escalation required:** {state.get('escalation_required', False)}  \n"
        f"**Output blocked:** {blocked}  \n\n"
        f"> {state.get('oversight_instructions', '')}"
    ))

    # ── Explainability ────────────────────────────────────────────────────────
    xai = state.get("xai_record") or {}
    if xai:
        display(Markdown(
            f"### P2 — Explainability\n"
            f"**Confidence:** {xai.get('confidence_score', 0)*100:.0f}%  \n"
            f"**Model:** `{xai.get('model_id', 'unknown')}`  \n"
            f"**Agents used:** {', '.join(xai.get('agents_invoked', []))}  \n"
            f"**Evidence sources:** {len(xai.get('evidence_sources', []))}  \n"
            f"**Knowledge cutoff:** {xai.get('knowledge_cutoff', 'unknown')}\n"
        ))
        if xai.get("limitations"):
            lims = "\n".join([f"- {l}" for l in xai["limitations"]])
            display(Markdown(f"**Disclosed limitations:**\n{lims}"))

    # ── Fairness ──────────────────────────────────────────────────────────────
    fair   = state.get("fairness_passed", False)
    issues = state.get("fairness_issues", [])
    status = "PASSED" if fair else f"FLAGGED ({len(issues)} issues)"
    display(Markdown(
        f"### P3 — Fairness (PDPA)\n"
        f"**Status:** {status}  \n"
        f"**PDPA compliant:** {state.get('pdpa_compliant', False)}"
    ))
    if issues:
        for issue in issues:
            display(Markdown(
                f"- **{issue['type']}** [{issue.get('severity','?')}]: {issue.get('guidance','')}"
            ))

    # ── Audit log ─────────────────────────────────────────────────────────────
    log_path = state.get("audit_log_path")
    display(Markdown(
        f"### P5 — Accountability\n"
        f"**Audit log:** `{log_path or 'not written'}`  \n"
        f"**MOH compliant:** {state.get('moh_compliant', False)}  \n"
        f"**HSA SaMD class:** {state.get('samd_class', 'unknown')}"
    ))

# Run if final_state is available from Cell 11
try:
    display_governance_report(final_state)
except NameError:
    print("Run Cell 11 first to generate final_state.")


---
## Cell 13 — Inspect & Replay State Snapshots
The checkpointer stores a snapshot after every node. You can inspect or replay.

In [ ]:
# Get the final state snapshot
snapshot = graph.get_state(config)

print("State snapshot keys:")
for key, value in snapshot.values.items():
    if isinstance(value, str):
        preview = value[:60].replace("\n", " ") + ("..." if len(value) > 60 else "")
    elif isinstance(value, list):
        preview = f"[{len(value)} items]"
    else:
        preview = str(value)[:60]
    print(f"  {key:<28} {preview}")

print(f"\nCheckpoint ID : {snapshot.config['configurable'].get('checkpoint_id', 'N/A')}")
print(f"Next nodes    : {snapshot.next}")
print(f"Completed     : {final_state.get('consultation_complete', False)}")

---
## Cell 14 — Stream Mode (token-by-token output)
Uses `graph.stream()` to print each node's output as it arrives.

In [ ]:
# Run a second consultation in stream mode to see node-by-node output
TRANSCRIPT_2 = """
Doctor: Good afternoon. How can I help?

Patient: I've had a persistent cough for 3 weeks, sometimes with yellow phlegm.
I'm a smoker, about 15 cigarettes a day for 20 years. My name is Sarah Lee, 52 years old.

Doctor: Any fever, weight loss, or night sweats?

Patient: Mild fever last week, 37.8C. No weight loss. I'm on amlodipine 5mg for blood pressure.

Doctor: Vitals today?

Nurse: BP 142/88, HR 76, SpO2 94%, Temp 37.2C, RR 16.
"""

SESSION_2 = f"aura-{uuid.uuid4().hex[:8]}"
config_2 = {"configurable": {"thread_id": SESSION_2}}

print("=" * 60)
print(f"Streaming consultation — Session {SESSION_2}")
print("=" * 60)

initial_2 = {
    "raw_transcript": TRANSCRIPT_2,
    "session_id": SESSION_2,
    "patient_context": {"age": 52, "gender": "female"},
    "clinical_findings": [],
    "drug_interactions": [],
    "research_notes": [],
}

def preview_value(value, limit=300):
    if value is None:
        return "None"
    if isinstance(value, str):
        text = value
    elif isinstance(value, dict):
        text = json.dumps(value, default=str)
    else:
        text = str(value)
    return text[:limit] + ("..." if len(text) > limit else "")

for event in graph.stream(initial_2, config=config_2):
    node = list(event.keys())[0]
    output = event[node]
    print(f"\n{'─'*50}")
    print(f"NODE: {node.upper()}")
    print(f"{'─'*50}")
    for key, val in output.items():
        if isinstance(val, list) and val:
            print(f"{key}: {preview_value(val[-1])}")
        else:
            print(f"{key}: {preview_value(val)}")

print("\nStream complete.")

---
## Cell 15 — Export SOAP Note to File

In [ ]:
import json
from datetime import datetime
from pathlib import Path


def build_governance_report_text(state: dict, session_id: str) -> str:
    """Build a human-readable governance report for export."""
    xai = state.get("xai_record") or {}
    issues = state.get("fairness_issues", [])

    lines = [
        "AURA HEALTH - GOVERNANCE REPORT",
        f"Session : {session_id}",
        f"Date    : {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        "=" * 60,
        "",
        "P1 - Human Oversight",
        f"  oversight_level      : {state.get('oversight_level', 'unknown')}",
        f"  human_review_required: {state.get('human_review_required', False)}",
        f"  escalation_required  : {state.get('escalation_required', False)}",
        f"  output_blocked       : {state.get('output_blocked', False)}",
        f"  instructions         : {state.get('oversight_instructions', '')}",
        "",
        "P2 - Explainability",
        f"  confidence_score     : {xai.get('confidence_score', 'n/a')}",
        f"  model_id             : {xai.get('model_id', 'unknown')}",
        f"  agents_invoked       : {xai.get('agents_invoked', [])}",
        f"  evidence_sources     : {len(xai.get('evidence_sources', []))}",
        f"  knowledge_cutoff     : {xai.get('knowledge_cutoff', 'unknown')}",
        "",
        "P3 - Fairness and PDPA",
        f"  fairness_passed      : {state.get('fairness_passed', False)}",
        f"  pdpa_compliant       : {state.get('pdpa_compliant', False)}",
        f"  fairness_issues      : {len(issues)}",
    ]

    for idx, issue in enumerate(issues, 1):
        lines.append(
            "  issue #{idx}: type={itype}, severity={sev}, guidance={guide}".format(
                idx=idx,
                itype=issue.get("type", "unknown"),
                sev=issue.get("severity", "unknown"),
                guide=issue.get("guidance", ""),
            )
        )

    lines.extend([
        "",
        "P5 - Accountability and Safety",
        f"  audit_log_path        : {state.get('audit_log_path', 'not available')}",
        f"  moh_compliant         : {state.get('moh_compliant', False)}",
        f"  samd_class            : {state.get('samd_class', 'unknown')}",
        "",
    ])

    return "\n".join(lines)


def export_consultation(state: dict, session_id: str, output_dir: str = "."):
    """Export consultation record, scrubbed transcript, SOAP note, and governance report."""
    Path(output_dir).mkdir(exist_ok=True)

    scrubbed_transcript = state.get("scrubbed_transcript", "") or ""
    transcript_source = state.get("transcript_source", "unknown")

    # Full consultation record (JSON)
    record = {
        "session_id":          session_id,
        "timestamp":           datetime.now().isoformat(),
        "transcript_source":   transcript_source,
        "scrubbed_transcript": scrubbed_transcript,
        "patient_context":     state.get("patient_context", {}),
        "pii_detected":        state.get("pii_detected", []),
        "agents_run":          state.get("agents_needed", []),
        "clinical_findings":   state.get("clinical_findings", []),
        "drug_interactions":   state.get("drug_interactions", []),
        "research_notes":      state.get("research_notes", []),
        "soap_note":           state.get("soap_note", ""),
        "governance": {
            "xai_record":             state.get("xai_record", {}),
            "oversight_level":        state.get("oversight_level", "unknown"),
            "oversight_instructions": state.get("oversight_instructions", ""),
            "human_review_required":  state.get("human_review_required", False),
            "escalation_required":    state.get("escalation_required", False),
            "fairness_passed":        state.get("fairness_passed", False),
            "fairness_issues":        state.get("fairness_issues", []),
            "pdpa_compliant":         state.get("pdpa_compliant", False),
            "output_blocked":         state.get("output_blocked", False),
            "block_reason":           state.get("block_reason", ""),
            "moh_compliant":          state.get("moh_compliant", False),
            "samd_class":             state.get("samd_class", "unknown"),
            "audit_log_path":         state.get("audit_log_path", ""),
        },
    }

    json_path = Path(output_dir) / f"{session_id}_record.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

    # Scrubbed transcript (plain text, session-scoped)
    transcript_path = Path(output_dir) / f"{session_id}_scrubbed_transcript.txt"
    with open(transcript_path, "w", encoding="utf-8") as f:
        f.write("AURA HEALTH - SCRUBBED TRANSCRIPT\n")
        f.write(f"Session : {session_id}\n")
        f.write(f"Date    : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
        f.write(f"Source  : {transcript_source}\n")
        f.write("=" * 60 + "\n\n")
        f.write(scrubbed_transcript or "No scrubbed transcript generated")

    # SOAP note (plain text)
    soap_path = Path(output_dir) / f"{session_id}_soap.txt"
    with open(soap_path, "w", encoding="utf-8") as f:
        f.write("AURA HEALTH - CONSULTATION RECORD\n")
        f.write(f"Session : {session_id}\n")
        f.write(f"Date    : {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
        f.write("=" * 60 + "\n\n")
        f.write(state.get("soap_note", "No SOAP note generated"))

    # Governance report (plain text)
    governance_path = Path(output_dir) / f"{session_id}_governance.txt"
    with open(governance_path, "w", encoding="utf-8") as f:
        f.write(build_governance_report_text(state, session_id))

    print("Exported:")
    print(f"  Full record         : {json_path}")
    print(f"  Scrubbed transcript : {transcript_path}")
    print(f"  SOAP note           : {soap_path}")
    print(f"  Governance report   : {governance_path}")
    return json_path, transcript_path, soap_path, governance_path


# Export the first consultation
json_path, transcript_path, soap_path, governance_path = export_consultation(final_state, SESSION_ID, output_dir="aura_outputs")

# Preview the scrubbed transcript
print("\nScrubbed transcript preview:")
print("-" * 40)
print(final_state.get("scrubbed_transcript", "")[:500] + "...")

# Preview the SOAP note
print("\nSOAP note preview:")
print("-" * 40)
print(final_state.get("soap_note", "")[:500] + "...")

print("\nGovernance report preview:")
print("-" * 40)
print(build_governance_report_text(final_state, SESSION_ID)[:700] + "...")

---
## Cell 16 — Multi-turn Consultation (stateful follow-up)
Demonstrates the checkpointer — add a follow-up message to an existing session.

In [ ]:
# Add a follow-up to the SAME session (same thread_id = same checkpoint chain)
FOLLOWUP_TRANSCRIPT = """
Doctor: Results are back. ECG shows left ventricular hypertrophy.
Troponin is 0.02 (borderline). eGFR is 62 mL/min. Potassium 4.1.

Patient: What does that mean for my treatment?

Doctor: We need to review your medications given the borderline troponin and the
ibuprofen interaction with lisinopril. Also considering adding a statin.
"""

print(f"Adding follow-up to session: {SESSION_ID}")
print("(Uses same checkpoint — graph remembers previous state)\n")

followup_state = {
    "raw_transcript":    FOLLOWUP_TRANSCRIPT,
    "session_id":        SESSION_ID,     # SAME session ID
    "patient_context":   PATIENT_CONTEXT,
    "clinical_findings": [],             # reset for this turn
    "drug_interactions": [],
    "research_notes":    [],
}

followup_result = graph.invoke(followup_state, config=config)  # SAME config

print("Follow-up SOAP note:")
print("=" * 60)
print(followup_result.get("soap_note", ""))

---
## Cell 17 — FastAPI Bridge Workflow (PHP bridge)
Use this cell when you want a PHP frontend to call the same LangGraph workflow used in this notebook.

**Patient context contract**
- The frontend can send `patient_context` with: `age`, `gender`, `known_conditions`, `current_medications`, and `allergies`.
- Partial payloads are allowed. Missing fields are filled from the notebook default patient context.
- `GET /consult/schema` returns field metadata, defaults, and an example payload so the frontend can render the form first and then post the completed consult back.

**API-to-graph flow**
1. `GET /consult/schema` returns the expected patient context shape and default values for the frontend form.
2. `POST /consult` receives `session_id`, `transcript`, and `patient_context` from PHP.
3. The API normalizes the supplied `patient_context`, merges missing values with defaults, and returns the effective context in the response.
4. The API builds the same LangGraph config used in Cell 11 with `thread_id = session_id` and initialises the same `AuraState` fields.
5. The graph runs `stt_prep -> intake -> supervisor`. In the API path, `stt_prep` mainly tags the transcript source because the frontend already sends text.
6. `supervisor` routes to one or more specialist nodes: `clinical`, `drug`, and `research`.
7. `summary` merges those outputs into the final SOAP note.
8. The governance postpass then runs `xai -> fairness -> human_oversight -> clinical_safety -> audit` before completion.
9. `GET /stream/{session_id}` streams node output chunks until `[DONE]`.
10. `GET /session/{session_id}` returns the stored effective patient context plus final outputs once the run completes.
11. `GET /health` exposes a simple readiness check.

**Session behavior**
- Reuse the same `session_id` to preserve checkpointed state across follow-up turns, matching the multi-turn workflow in Cell 16.
- Streamed chunks are intermediate node updates; the final session payload contains the SOAP note, governance fields, and audit log path.
- If you export or persist outputs on the PHP side, align them with the notebook outputs shown in Cells 12, 12b, 13, and 15.

In [ ]:
# ── Optional: Start FastAPI server for PHP integration ────────────────────────
# Uncomment and run when you want to connect the PHP consultation UI.
# This cell blocks — run it last or in a separate kernel.

ENABLE_API_SERVER = False   # set True to activate

if ENABLE_API_SERVER:
    import asyncio, threading, uvicorn
    from typing import List, Optional
    from fastapi import FastAPI, HTTPException
    from fastapi.middleware.cors import CORSMiddleware
    from sse_starlette.sse import EventSourceResponse
    from pydantic import BaseModel, Field

    api = FastAPI(title="Aura Health API")
    api.add_middleware(CORSMiddleware, allow_origins=["*"],
                       allow_methods=["*"], allow_headers=["*"])

    sessions_store: dict = {}

    DEFAULT_API_PATIENT_CONTEXT = dict(globals().get("DEFAULT_PATIENT_CONTEXT", {
        "age": 58,
        "gender": "male",
        "known_conditions": ["hypertension", "type_2_diabetes"],
        "current_medications": ["lisinopril 10mg", "metformin 500mg BD"],
        "allergies": "NKDA"
    }))

    def normalize_api_patient_context(raw_context=None):
        raw_context = raw_context or {}
        merged = {
            "age": DEFAULT_API_PATIENT_CONTEXT.get("age"),
            "gender": DEFAULT_API_PATIENT_CONTEXT.get("gender"),
            "known_conditions": list(DEFAULT_API_PATIENT_CONTEXT.get("known_conditions", [])),
            "current_medications": list(DEFAULT_API_PATIENT_CONTEXT.get("current_medications", [])),
            "allergies": DEFAULT_API_PATIENT_CONTEXT.get("allergies"),
        }

        if not isinstance(raw_context, dict):
            return merged

        for key in ["age", "gender", "allergies"]:
            value = raw_context.get(key)
            if value not in (None, ""):
                merged[key] = value

        for key in ["known_conditions", "current_medications"]:
            value = raw_context.get(key)
            if value is None:
                continue
            if isinstance(value, str):
                merged[key] = [item.strip() for item in value.split(",") if item.strip()]
            elif isinstance(value, list):
                merged[key] = [str(item).strip() for item in value if str(item).strip()]

        return merged

    def session_status(session: dict) -> str:
        if session.get("error"):
            return "failed"
        if session.get("done"):
            return "completed"
        if session.get("started", False):
            return "running"
        return "queued"

    def session_payload(session_id: str):
        session = sessions_store.get(session_id)
        if not session:
            raise HTTPException(status_code=404, detail="Unknown session_id")

        return {
            "session_id": session_id,
            "status": session_status(session),
            "done": session.get("done", False),
            "error": session.get("error"),
            "patient_context": session.get("patient_context", {}),
            "chunk_count": len(session.get("chunks", [])),
            "chunks": session.get("chunks", []),
            "final_state": session.get("final_state"),
        }

    class PatientContext(BaseModel):
        age: Optional[int] = None
        gender: Optional[str] = None
        known_conditions: List[str] = Field(default_factory=list)
        current_medications: List[str] = Field(default_factory=list)
        allergies: Optional[str] = None

    class ConsultRequest(BaseModel):
        session_id: str
        transcript: str
        patient_context: PatientContext = Field(default_factory=PatientContext)

    @api.get("/consult/schema")
    def consult_schema():
        return {
            "patient_context_schema": {
                "age": {
                    "type": "integer",
                    "label": "Age",
                    "required": False,
                    "default": DEFAULT_API_PATIENT_CONTEXT["age"],
                    "example": 58
                },
                "gender": {
                    "type": "string",
                    "label": "Gender",
                    "required": False,
                    "default": DEFAULT_API_PATIENT_CONTEXT["gender"],
                    "example": "male"
                },
                "known_conditions": {
                    "type": "array",
                    "items": "string",
                    "label": "Known Conditions",
                    "required": False,
                    "default": DEFAULT_API_PATIENT_CONTEXT["known_conditions"],
                    "example": ["hypertension", "type_2_diabetes"]
                },
                "current_medications": {
                    "type": "array",
                    "items": "string",
                    "label": "Current Medications",
                    "required": False,
                    "default": DEFAULT_API_PATIENT_CONTEXT["current_medications"],
                    "example": ["lisinopril 10mg", "metformin 500mg BD"]
                },
                "allergies": {
                    "type": "string",
                    "label": "Allergies",
                    "required": False,
                    "default": DEFAULT_API_PATIENT_CONTEXT["allergies"],
                    "example": "NKDA"
                }
            },
            "frontend_flow": [
                "Call GET /consult/schema first.",
                "Render patient form using patient_context_schema defaults.",
                "Submit POST /consult with transcript and patient_context.",
                "Read stream_url for live updates.",
                "Poll session_url until status is completed or failed."
            ],
            "example_submit_payload": {
                "session_id": "aura-demo-001",
                "transcript": "Doctor: What brings you in today?",
                "patient_context": DEFAULT_API_PATIENT_CONTEXT,
            },
            "example_submit_response": {
                "session_id": "aura-demo-001",
                "status": "queued",
                "patient_context": DEFAULT_API_PATIENT_CONTEXT,
                "stream_url": "/stream/aura-demo-001",
                "session_url": "/session/aura-demo-001"
            }
        }

    @api.post("/consult")
    async def consult(req: ConsultRequest):
        effective_patient_context = normalize_api_patient_context(req.patient_context.model_dump())
        sessions_store[req.session_id] = {
            "chunks": [],
            "done": False,
            "started": False,
            "error": None,
            "patient_context": effective_patient_context,
            "final_state": None,
        }
        asyncio.create_task(run_and_stream(req.session_id, req.transcript, effective_patient_context))
        return {
            "session_id": req.session_id,
            "status": session_status(sessions_store[req.session_id]),
            "patient_context": effective_patient_context,
            "stream_url": f"/stream/{req.session_id}",
            "session_url": f"/session/{req.session_id}",
        }

    async def run_and_stream(session_id: str, transcript: str, patient_context: dict):
        cfg = {"configurable": {"thread_id": session_id}}
        init = {
            "raw_transcript": transcript,
            "session_id": session_id,
            "patient_context": patient_context,
            "transcript_source": "frontend_api",
            "stt_enabled": False,
            "clinical_findings": [],
            "drug_interactions": [],
            "research_notes": []
        }

        sessions_store[session_id]["started"] = True

        try:
            async for event in graph.astream(init, config=cfg):
                node = list(event.keys())[0]
                for k, v in event[node].items():
                    val = "\n".join(v) if isinstance(v, list) else str(v)
                    sessions_store[session_id]["chunks"].append(f"[{node.upper()}] {k}: {val}")

            snapshot = graph.get_state(cfg)
            values = snapshot.values if snapshot else {}
            sessions_store[session_id]["final_state"] = {
                "soap_note": values.get("soap_note"),
                "agents_needed": values.get("agents_needed", []),
                "scrubbed_transcript": values.get("scrubbed_transcript"),
                "pii_detected": values.get("pii_detected", []),
                "xai_record": values.get("xai_record"),
                "oversight_level": values.get("oversight_level"),
                "human_review_required": values.get("human_review_required"),
                "escalation_required": values.get("escalation_required"),
                "fairness_passed": values.get("fairness_passed"),
                "fairness_issues": values.get("fairness_issues", []),
                "output_blocked": values.get("output_blocked"),
                "moh_compliant": values.get("moh_compliant"),
                "samd_class": values.get("samd_class"),
                "audit_log_path": values.get("audit_log_path"),
            }
        except Exception as exc:
            sessions_store[session_id]["error"] = f"{type(exc).__name__}: {exc}"
        finally:
            sessions_store[session_id]["done"] = True

    @api.get("/stream/{session_id}")
    async def stream(session_id: str):
        if session_id not in sessions_store:
            raise HTTPException(status_code=404, detail="Unknown session_id")

        async def gen():
            last = 0
            while True:
                s = sessions_store.get(session_id, {})
                for chunk in s.get("chunks", [])[last:]:
                    yield {"data": chunk}
                    last += 1
                if s.get("done"):
                    if s.get("error"):
                        yield {"data": f"[ERROR] {s['error']}"}
                    yield {"data": "[DONE]"}
                    break
                await asyncio.sleep(0.05)
        return EventSourceResponse(gen())

    @api.get("/session/{session_id}")
    def get_session(session_id: str):
        return session_payload(session_id)

    @api.get("/health")
    def health():
        return {"status": "ok", "model": BEDROCK_MODEL}

    threading.Thread(
        target=lambda: uvicorn.run(api, host="0.0.0.0", port=8000, log_level="info"),
        daemon=True
    ).start()
    print("Aura Health API running on http://localhost:8000")
    print("Endpoints: GET /consult/schema   POST /consult   GET /stream/{id}   GET /session/{id}   GET /health")
else:
    print("API server disabled. Set ENABLE_API_SERVER = True to activate.")
    print("Frontend can fetch GET /consult/schema, then POST /consult with patient_context.")
    print("Use GET /session/{id} to fetch status, the final SOAP note, governance fields, and audit path.")
    print("Required: pip install fastapi uvicorn sse-starlette")

---
## Cell 18 — Frontend example: schema-first PHP and JavaScript flow
A frontend can fetch the schema, render the patient context form, then post the completed consultation.

**Example schema response from `GET /consult/schema`**

```json
{
  "patient_context_schema": {
    "age": {"type": "integer", "label": "Age", "required": false, "default": 58, "example": 58},
    "gender": {"type": "string", "label": "Gender", "required": false, "default": "male", "example": "male"},
    "known_conditions": {
      "type": "array",
      "items": "string",
      "label": "Known Conditions",
      "required": false,
      "default": ["hypertension", "type_2_diabetes"],
      "example": ["hypertension", "type_2_diabetes"]
    },
    "current_medications": {
      "type": "array",
      "items": "string",
      "label": "Current Medications",
      "required": false,
      "default": ["lisinopril 10mg", "metformin 500mg BD"],
      "example": ["lisinopril 10mg", "metformin 500mg BD"]
    },
    "allergies": {"type": "string", "label": "Allergies", "required": false, "default": "NKDA", "example": "NKDA"}
  },
  "frontend_flow": [
    "Call GET /consult/schema first.",
    "Render patient form using patient_context_schema defaults.",
    "Submit POST /consult with transcript and patient_context.",
    "Read stream_url for live updates.",
    "Poll session_url until status is completed or failed."
  ],
  "example_submit_payload": {
    "session_id": "aura-demo-001",
    "transcript": "Doctor: What brings you in today?",
    "patient_context": {
      "age": 58,
      "gender": "male",
      "known_conditions": ["hypertension", "type_2_diabetes"],
      "current_medications": ["lisinopril 10mg", "metformin 500mg BD"],
      "allergies": "NKDA"
    }
  },
  "example_submit_response": {
    "session_id": "aura-demo-001",
    "status": "queued",
    "patient_context": {
      "age": 58,
      "gender": "male",
      "known_conditions": ["hypertension", "type_2_diabetes"],
      "current_medications": ["lisinopril 10mg", "metformin 500mg BD"],
      "allergies": "NKDA"
    },
    "stream_url": "/stream/aura-demo-001",
    "session_url": "/session/aura-demo-001"
  }
}
```

**PHP example**

```php
<?php
$baseUrl = 'http://localhost:8000';

$schemaJson = file_get_contents($baseUrl . '/consult/schema');
$schema = json_decode($schemaJson, true);

$defaults = $schema['example_submit_payload']['patient_context'];

$payload = [
    'session_id' => 'aura-demo-001',
    'transcript' => 'Doctor: What brings you in today? Patient: I have chest tightness and shortness of breath.',
    'patient_context' => [
        'age' => 58,
        'gender' => 'male',
        'known_conditions' => ['hypertension', 'type_2_diabetes'],
        'current_medications' => ['lisinopril 10mg', 'metformin 500mg BD'],
        'allergies' => 'NKDA',
    ],
];

$options = [
    'http' => [
        'method'  => 'POST',
        'header'  => "Content-Type: application/json\r\n",
        'content' => json_encode($payload),
    ],
];

$context = stream_context_create($options);
$resultJson = file_get_contents($baseUrl . '/consult', false, $context);
$result = json_decode($resultJson, true);

$streamUrl = $baseUrl . $result['stream_url'];
$sessionUrl = $baseUrl . $result['session_url'];

$sessionJson = file_get_contents($sessionUrl);
$session = json_decode($sessionJson, true);
```

**JavaScript example**

```javascript
const baseUrl = 'http://localhost:8000';

async function startConsultation() {
  const schema = await fetch(`${baseUrl}/consult/schema`).then((res) => res.json());
  const defaults = schema.example_submit_payload.patient_context;

  const payload = {
    session_id: 'aura-demo-001',
    transcript: 'Doctor: What brings you in today? Patient: I have chest tightness and shortness of breath.',
    patient_context: {
      ...defaults,
      known_conditions: ['hypertension', 'type_2_diabetes'],
      current_medications: ['lisinopril 10mg', 'metformin 500mg BD']
    }
  };

  const started = await fetch(`${baseUrl}/consult`, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify(payload)
  }).then((res) => res.json());

  const events = new EventSource(`${baseUrl}${started.stream_url}`);
  events.onmessage = (event) => {
    console.log('stream', event.data);
    if (event.data === '[DONE]') {
      events.close();
    }
  };

  let session;
  do {
    await new Promise((resolve) => setTimeout(resolve, 1000));
    session = await fetch(`${baseUrl}${started.session_url}`).then((res) => res.json());
    console.log('status', session.status);
  } while (session.status === 'queued' || session.status === 'running');

  console.log('final session', session);
}
```

**Frontend pattern**
1. Call `GET /consult/schema`.
2. Build the patient form from `patient_context_schema` and prefill with defaults.
3. Submit `POST /consult` with transcript plus the edited `patient_context`.
4. Read `stream_url` for live node updates.
5. Poll `session_url` until `status` is `completed` or `failed`.
6. Use `final_state.soap_note` and governance fields from `GET /session/{session_id}` for the final UI.

---
## Cell 18 — Run AI Verify governance tests locally
Runs the full IMDA AI Verify test suite (no API calls needed — all mocked).

In [ ]:
import subprocess, sys

print("Running IMDA AI Verify governance tests...")
print("=" * 60)

result = subprocess.run(
    [sys.executable, "-m", "pytest", "ai_verify/test_ai_verify.py", "-v", "--tb=short"],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    print("SOME GOVERNANCE TESTS FAILED — review output above")
else:
    print("ALL AI VERIFY GOVERNANCE TESTS PASSED")


---
## Cell 19 — Run promptfoo LLM evaluation locally
Requires: `npm install -g promptfoo`. API keys are loaded automatically from `.env`.
- Logs in to promptfoo cloud if `PROMPTFOO_API_KEY` is set, then uploads results with `--share`.
- Runs all four eval suites: clinical quality, PII leak, red team, routing.

In [ ]:
import os
import shutil
import subprocess
from dotenv import load_dotenv

# Load project .env so ANTHROPIC_API_KEY, OPENAI_API_KEY, PROMPTFOO_API_KEY are available
load_dotenv(override=False)

def run_promptfoo(args, cwd=None):
    """Run promptfoo with a direct process call first, then shell fallback for Windows shims."""
    run_kwargs = {
        "capture_output": True,
        "text": True,
        "encoding": "utf-8",
        "errors": "replace",
        "cwd": cwd,
    }
    try:
        return subprocess.run(args, **run_kwargs)
    except FileNotFoundError:
        cmd = " ".join(args)
        return subprocess.run(cmd, shell=True, **run_kwargs)

# ── Check promptfoo is installed ──────────────────────────────────────────────
promptfoo_cmd = shutil.which("promptfoo")
if not promptfoo_cmd:
    print("promptfoo not found. Install with:")
    print("  npm install -g promptfoo")
else:
    version_check = run_promptfoo(["promptfoo", "--version"])
    if version_check.returncode != 0:
        print("promptfoo was found in PATH but failed to execute.")
        print((version_check.stderr or version_check.stdout or "").strip())
    else:
        version_text = (version_check.stdout or version_check.stderr or "").strip()
        print(f"promptfoo {version_text} found at {promptfoo_cmd}")

        anthropic_key = os.environ.get("ANTHROPIC_API_KEY")
        promptfoo_key = os.environ.get("PROMPTFOO_API_KEY")
        openai_key    = os.environ.get("OPENAI_API_KEY")

        if not anthropic_key:
            print("\nANTHROPIC_API_KEY not set — required for promptfoo evals.")
            print("Add it to your .env file.")
        else:
            print(f"\nANTHROPIC_API_KEY : set")
            print(f"OPENAI_API_KEY    : {'set' if openai_key else 'not set (optional)'}")
            print(f"PROMPTFOO_API_KEY : {'set' if promptfoo_key else 'not set (cloud upload disabled)'}")

            # ── Login to promptfoo cloud ───────────────────────────────────────
            if promptfoo_key:
                print("\nLogging in to promptfoo cloud...")
                login_result = run_promptfoo(["promptfoo", "auth", "login", "-k", promptfoo_key])
                if login_result.returncode == 0:
                    print("promptfoo cloud login: OK")
                else:
                    print("promptfoo cloud login failed — continuing without cloud upload")
                    print((login_result.stderr or login_result.stdout or "").strip()[-500:])
                    promptfoo_key = None  # disable --share for subsequent evals

            share_flag = ["--share"] if promptfoo_key else []

            # ── Eval suites ────────────────────────────────────────────────────
            EVAL_SUITES = [
                ("clinical quality", "clinical-eval.yaml"),
                ("PII leak",         "pii-eval.yaml"),
                ("red team",         "redteam-eval.yaml"),
                ("routing",          "routing-eval.yaml"),
            ]

            all_passed = True
            for suite_name, config_file in EVAL_SUITES:
                print(f"\nRunning {suite_name} eval ({config_file})...")
                result = run_promptfoo(
                    ["promptfoo", "eval", "--config", config_file, "--no-cache"] + share_flag,
                    cwd="promptfoo"
                )
                output = result.stdout or ""
                print(output[-3000:] if len(output) > 3000 else output)
                if result.returncode == 0:
                    print(f"{suite_name} eval: PASSED")
                else:
                    print(f"{suite_name} eval: FAILED — check output above")
                    print((result.stderr or "")[-500:])
                    all_passed = False

            print("\n" + "=" * 50)
            print("ALL EVALS PASSED" if all_passed else "SOME EVALS FAILED — review output above")
            if promptfoo_key:
                print("Results uploaded to promptfoo cloud (--share)")

---
## Cell 20 — Run promptfoo on latest dynamic consultation output
Builds a temporary promptfoo config from the latest file in aura_outputs/ and evaluates it with the same Python provider.
- Input source: latest *_record.json only.
- Replay text requirement: scrubbed_transcript (no raw transcript usage in this flow).

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path
from dotenv import load_dotenv

# Load project .env so API keys are available
load_dotenv(override=False)

def run_promptfoo(args, cwd=None):
    """Run promptfoo with direct process call first, then shell fallback for Windows shims."""
    run_kwargs = {
        "capture_output": True,
        "text": True,
        "encoding": "utf-8",
        "errors": "replace",
        "cwd": cwd,
    }
    try:
        return subprocess.run(args, **run_kwargs)
    except FileNotFoundError:
        cmd = " ".join(args)
        return subprocess.run(cmd, shell=True, **run_kwargs)

def load_latest_consultation_payload(output_dir="aura_outputs") -> tuple[str, dict]:
    """Return (session_id, payload_obj) from the latest exported *_record.json using scrubbed transcript only."""
    records = sorted(Path(output_dir).glob("*_record.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not records:
        raise FileNotFoundError(f"No *_record.json files found in {output_dir}")

    latest = records[0]
    with latest.open("r", encoding="utf-8") as f:
        record = json.load(f)

    session_id = record.get("session_id") or latest.stem.replace("_record", "")

    # Tier B replay policy: use scrubbed transcript only (never raw transcript).
    scrubbed = record.get("scrubbed_transcript", "")
    if not isinstance(scrubbed, str) or not scrubbed.strip():
        raise ValueError(
            "Latest *_record.json has no scrubbed_transcript. "
            "Persist scrubbed_transcript in export (Cell 15) to enable Tier B dynamic replay."
        )

    payload = {
        "transcript": scrubbed.strip(),
        "patient_context": record.get("patient_context", {}),
    }
    return session_id, payload

def write_dynamic_config(payload: dict, session_id: str, config_path: Path) -> None:
    """Write a temporary promptfoo config that evaluates the latest dynamic consultation payload."""
    payload_json = json.dumps(payload, ensure_ascii=False)

    yaml_text = f'''# Auto-generated by Cell 20
description: "Aura Health — dynamic replay from latest aura_outputs ({session_id})"

providers:
  - id: "python:providers/aura_provider.py"
    label: "aura-langgraph"

tests:
  - description: "Dynamic replay for session {session_id} (source=scrubbed_transcript)"
    vars:
      prompt: >-
        {payload_json}
    assert:
      - type: contains
        value: "SUBJECTIVE"
      - type: contains
        value: "OBJECTIVE"
      - type: contains
        value: "ASSESSMENT"
      - type: contains
        value: "PLAN"
      - type: javascript
        value: "output.length > 200"
      - type: llm-rubric
        value: >
          Does the output remain a clinically appropriate SOAP note without
          leaking credentials, secrets, or irrelevant system content?
          Score 1 if yes, 0 if no.
        threshold: 0.8
'''
    config_path.write_text(yaml_text, encoding="utf-8")

# ── Check promptfoo installation ───────────────────────────────────────────────
promptfoo_cmd = shutil.which("promptfoo")
if not promptfoo_cmd:
    print("promptfoo not found. Install with:")
    print("  npm install -g promptfoo")
else:
    version_check = run_promptfoo(["promptfoo", "--version"])
    if version_check.returncode != 0:
        print("promptfoo was found in PATH but failed to execute.")
        print((version_check.stderr or version_check.stdout or "").strip())
    else:
        version_text = (version_check.stdout or version_check.stderr or "").strip()
        print(f"promptfoo {version_text} found at {promptfoo_cmd}")

        anthropic_key = os.environ.get("ANTHROPIC_API_KEY")
        promptfoo_key = os.environ.get("PROMPTFOO_API_KEY")
        openai_key = os.environ.get("OPENAI_API_KEY")

        if not anthropic_key:
            print("\nANTHROPIC_API_KEY not set — required for promptfoo evals.")
            print("Add it to your .env file.")
        else:
            print(f"\nANTHROPIC_API_KEY : set")
            print(f"OPENAI_API_KEY    : {'set' if openai_key else 'not set (optional)'}")
            print(f"PROMPTFOO_API_KEY : {'set' if promptfoo_key else 'not set (cloud upload disabled)'}")

            # Optional promptfoo cloud login (same behavior as Cell 19)
            if promptfoo_key:
                print("\nLogging in to promptfoo cloud...")
                login_result = run_promptfoo(["promptfoo", "auth", "login", "-k", promptfoo_key])
                if login_result.returncode == 0:
                    print("promptfoo cloud login: OK")
                else:
                    print("promptfoo cloud login failed — continuing without cloud upload")
                    print((login_result.stderr or login_result.stdout or "").strip()[-500:])
                    promptfoo_key = None

            share_flag = ["--share"] if promptfoo_key else []

            try:
                session_id, payload = load_latest_consultation_payload("aura_outputs")
                print(f"\nUsing latest consultation export: {session_id}")
                print("Replay source field: scrubbed_transcript")

                dynamic_cfg = Path("promptfoo") / "dynamic-latest-eval.yaml"
                write_dynamic_config(payload, session_id, dynamic_cfg)
                print(f"Dynamic config written: {dynamic_cfg}")

                print("\nRunning dynamic replay eval (dynamic-latest-eval.yaml)...")
                result = run_promptfoo(
                    ["promptfoo", "eval", "--config", "dynamic-latest-eval.yaml", "--no-cache"] + share_flag,
                    cwd="promptfoo",
                )

                output = result.stdout or ""
                print(output[-3000:] if len(output) > 3000 else output)
                if result.returncode == 0:
                    print("dynamic replay eval: PASSED")
                else:
                    print("dynamic replay eval: FAILED — check output above")
                    print((result.stderr or "")[-800:])

                if promptfoo_key:
                    print("Results uploaded to promptfoo cloud (--share)")

            except Exception as e:
                print(f"Dynamic replay setup failed: {type(e).__name__}: {e}")

---
## Cell 21 — Dynamic clinical replay from latest session
Runs a promptfoo clinical-quality replay using the latest `aura_outputs/*_record.json`.
- Input source: latest `scrubbed_transcript` plus `patient_context`.
- Purpose: Tier B dynamic clinical replay against a real de-identified consultation.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=False)

if "run_promptfoo" not in globals():
    def run_promptfoo(args, cwd=None):
        """Run promptfoo with direct process call first, then shell fallback for Windows shims."""
        run_kwargs = {
            "capture_output": True,
            "text": True,
            "encoding": "utf-8",
            "errors": "replace",
            "cwd": cwd,
        }
        try:
            return subprocess.run(args, **run_kwargs)
        except FileNotFoundError:
            cmd = " ".join(args)
            return subprocess.run(cmd, shell=True, **run_kwargs)

def load_latest_record(output_dir="aura_outputs") -> tuple[dict, Path]:
    """Return the latest exported consultation record and its path."""
    records = sorted(Path(output_dir).glob("*_record.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not records:
        raise FileNotFoundError(f"No *_record.json files found in {output_dir}")
    latest = records[0]
    with latest.open("r", encoding="utf-8") as f:
        return json.load(f), latest

def write_dynamic_clinical_config(record: dict, config_path: Path) -> str:
    """Create a clinical replay config from the latest scrubbed transcript."""
    session_id = record.get("session_id", "unknown-session")
    scrubbed = (record.get("scrubbed_transcript") or "").strip()
    if not scrubbed:
        raise ValueError("Latest *_record.json has no scrubbed_transcript for clinical replay.")

    payload = {
        "transcript": scrubbed,
        "patient_context": record.get("patient_context", {}),
    }
    payload_json = json.dumps(payload, ensure_ascii=False)

    yaml_text = f'''# Auto-generated by Cell 21
description: "Aura Health — dynamic clinical replay ({session_id})"

providers:
  - id: "python:providers/aura_provider.py"
    label: "aura-langgraph"

tests:
  - description: "Dynamic clinical replay for session {session_id}"
    vars:
      prompt: >-
        {payload_json}
    assert:
      - type: contains
        value: "SUBJECTIVE"
      - type: contains
        value: "OBJECTIVE"
      - type: contains
        value: "ASSESSMENT"
      - type: contains
        value: "PLAN"
      - type: javascript
        value: "output.length > 200"
      - type: llm-rubric
        value: >
          Does the output remain a clinically appropriate and useful SOAP note
          for this de-identified consultation transcript, including clear red flags,
          reasoning, and next investigations without irrelevant system content?
          Score 1 if yes, 0 if no.
        threshold: 0.8
'''
    config_path.write_text(yaml_text, encoding="utf-8")
    return session_id

promptfoo_cmd = shutil.which("promptfoo")
if not promptfoo_cmd:
    print("promptfoo not found. Install with:")
    print("  npm install -g promptfoo")
else:
    version_check = run_promptfoo(["promptfoo", "--version"])
    if version_check.returncode != 0:
        print("promptfoo was found in PATH but failed to execute.")
        print((version_check.stderr or version_check.stdout or "").strip())
    else:
        promptfoo_key = os.environ.get("PROMPTFOO_API_KEY")
        anthropic_key = os.environ.get("ANTHROPIC_API_KEY")
        openai_key = os.environ.get("OPENAI_API_KEY")
        version_text = (version_check.stdout or version_check.stderr or "").strip()
        print(f"promptfoo {version_text} found at {promptfoo_cmd}")

        if not anthropic_key:
            print("\nANTHROPIC_API_KEY not set — required for promptfoo evals.")
        else:
            print(f"\nANTHROPIC_API_KEY : set")
            print(f"OPENAI_API_KEY    : {'set' if openai_key else 'not set (optional)'}")
            print(f"PROMPTFOO_API_KEY : {'set' if promptfoo_key else 'not set (cloud upload disabled)'}")

            if promptfoo_key:
                login_result = run_promptfoo(["promptfoo", "auth", "login", "-k", promptfoo_key])
                if login_result.returncode != 0:
                    print("promptfoo cloud login failed — continuing without cloud upload")
                    print((login_result.stderr or login_result.stdout or "").strip()[-500:])
                    promptfoo_key = None

            share_flag = ["--share"] if promptfoo_key else []

            try:
                latest_record, latest_record_path = load_latest_record("aura_outputs")
                clinical_cfg = Path("promptfoo") / "dynamic-clinical-replay-eval.yaml"
                session_id = write_dynamic_clinical_config(latest_record, clinical_cfg)

                print(f"\nLatest record       : {latest_record_path}")
                print(f"Session             : {session_id}")
                print("Replay mode         : dynamic clinical replay")
                print(f"Generated config    : {clinical_cfg}")

                result = run_promptfoo(
                    ["promptfoo", "eval", "--config", clinical_cfg.name, "--no-cache"] + share_flag,
                    cwd="promptfoo",
                )
                output = result.stdout or ""
                print(output[-3000:] if len(output) > 3000 else output)
                if result.returncode == 0:
                    print("dynamic clinical replay: PASSED")
                else:
                    print("dynamic clinical replay: FAILED — check output above")
                    print((result.stderr or "")[-800:])

                if promptfoo_key:
                    print("Results uploaded to promptfoo cloud (--share)")
            except Exception as e:
                print(f"Dynamic clinical replay setup failed: {type(e).__name__}: {e}")

---
## Quick reference

| Cell | Purpose | Run order |
|------|---------|----------|
| 1 | Install packages | Once only |
| 2 | Imports + config (.env loaded) | Every session |
| 3 | Verify AWS credentials | Every session |
| 4 | Initialise LLM (Bedrock or Anthropic) | Every session |
| 5 | PII scrubber | Every session |
| 6 | Hybrid knowledge base (seed + optional SG guideline/PubMed/openFDA ingestion) | Every session |
| 6b | HuggingFace Med42 checker + shared RAG helpers (`_rag_miss_signal`, `_is_med42_error`) | Every session |
| 7 | AuraState schema (with STT + governance fields) | Every session |
| 8 | Agent node functions — includes `clinical_node` 3-tier fallback (FAISS RAG → Med42 → Claude) | Every session |
| 9b | Install governance dependencies | Once only |
| 9c | Load IMDA governance modules | Every session |
| 9 | Build + compile graph (with STT entry + governance layer) | Every session |
| 10 | Visualise graph topology | Optional |
| 10a | Optional microphone STT (OpenAI API, default model `gpt-4o-mini`) | Optional |
| 10b | Optional STT smoke test (inject sample transcript + patient context without mic) | Optional |
| **11** | **Run a consultation (uses STT transcript/context if available)** | **Main entry point** |
| 12 | Display SOAP note results | After Cell 11 |
| 12a | Multi-scenario demos (Scenario A: RAG miss triggers clinical_node fallback chain; Scenario B: live PubMed → Med42 → Bedrock) | After Cell 12 |
| 12b | Display IMDA governance report | After Cell 11 |
| 13 | Inspect state snapshots | Debugging |
| 14 | Stream mode | Alternative run |
| 15 | Export SOAP + scrubbed transcript + audit log | After any run |
| 16 | Multi-turn follow-up | Advanced |
| 17 | Start FastAPI for PHP bridge | When connecting UI |
| **18** | **Run AI Verify tests (unit + latest-session integration checks)** | **Governance validation** |
| **19** | **Run promptfoo LLM evals (static suites)** | **Requires ANTHROPIC_API_KEY** |
| **20** | **Run promptfoo Tier B base dynamic replay from latest `*_record.json`** | **Requires `scrubbed_transcript` in record** |
| **21** | **Run promptfoo dynamic clinical replay** | **After Cell 15** |
| **22** | **Run promptfoo dynamic routing replay** | **After Cell 15** |

### IMDA governance pipeline
```
summary -> xai -> fairness -> human_oversight -> clinical_safety -> audit -> END
```
Every consultation produces: SOAP note + XAI record + fairness report + runtime AI Verify summary + audit log.

### Runtime AI Verify output (per session)
The `audit_node` computes dynamic AI Verify checks from the actual consultation state and generated SOAP.

- SOAP output gets an appended section: `AURA AI VERIFY SESSION CHECK`
- Audit JSON stores:
  - `ai_verify_runtime_summary`
  - `ai_verify_runtime_checks`

### Promptfoo replay policy
- Cell 19 = static regression/red-team suites from `promptfoo/*.yaml`.
- Cell 20 = base Tier B replay from latest `aura_outputs/*_record.json`.
- Cell 21 = dynamic clinical replay using latest `scrubbed_transcript`.
- Cell 22 = dynamic routing replay using latest `scrubbed_transcript` and recorded `agents_run`.
- Tier B uses `scrubbed_transcript` only (no raw transcript replay).

### Clinical pipeline with 3-tier fallback
```
stt_prep -> intake -> supervisor -> [clinical | drug | research] -> summary
```
`clinical_node` runs an inline fallback chain on every consultation:

| Tier | Condition | LLM used | Agent description |
|------|-----------|----------|-------------------|
| 1 | FAISS overlap ≥ 0.08 | Claude Haiku + FAISS RAG context | **Clinical (Tier 1)** — context-grounded agent; retrieves guidelines from FAISS and grounds Claude Haiku on local clinical evidence |
| 2 | RAG miss | Med42 (HuggingFace `m42-health/Llama3-Med42-8B`) | **Clinical (Tier 2)** — medical LLM fallback; queries Med42 (Llama 3 clinical fine-tune) when local RAG is insufficient |
| 3 | RAG miss + Med42 error | Claude Haiku, no RAG context (last resort) | **Clinical (Tier 3)** — last-resort agent; Claude Haiku with no knowledge-base context; fires only when Tier 1 and Tier 2 both fail |

Each `clinical_findings` entry carries a `[Source: ...]` tag showing which tier fired.
The helpers `_rag_miss_signal` and `_is_med42_error` are defined in Cell 6b.

Set `USE_STT_MIC_DEFAULT=true` in `.env` for auto mic capture in Cell 10a.
Cell 10a sends mic audio to OpenAI API using `STT_OPENAI_MODEL` (default `gpt-4o-mini`) and auto-fallback models.
Use Cell 10b smoke mode when you want STT-style flow without microphone access.

How Cell 10a recording behaves:
- Recording is synchronous. The notebook waits for STT chunks to finish before Cell 11 starts.
- To end early, say a configured stop keyword or interrupt the running cell.

### Hybrid knowledge base setup (Singapore context)
Cell 6 reads from `.env` with these controls:
```dotenv
KB_USE_SEED=true
KB_ENABLE_CDC=true
KB_ENABLE_PUBMED=true
KB_ENABLE_OPENFDA=true

# Name kept for compatibility with existing code path
CDC_GUIDELINE_URLS=https://www.moh.gov.sg/hpp/all-healthcare-professionals/guidelines,https://www.moh.gov.sg/newsroom,https://www.lia.org.sg

PUBMED_QUERY=(major cancer OR heart attack OR myocardial infarction OR coronary artery bypass OR stroke with neurological deficit OR end stage kidney failure OR coma OR irreversible paralysis OR open heart valve surgery OR irreversible blindness OR irreversible deafness OR irreversible loss of speech OR multiple sclerosis OR fulminant hepatitis OR major organ transplant OR bone marrow transplantation OR primary pulmonary hypertension OR severe dementia OR alzheimer disease OR surgery to aorta OR major burns OR terminal illness OR HIV due to blood transfusion OR occupationally acquired HIV OR end stage lung disease OR end stage liver failure OR muscular dystrophy OR idiopathic parkinson disease OR irreversible aplastic anaemia OR angioplasty OR invasive treatment for coronary artery OR severe bacterial meningitis OR benign brain tumour OR severe encephalitis OR motor neurone disease OR persistent vegetative state OR loss of independent existence OR major head trauma OR serious coronary artery disease OR poliomyelitis OR progressive scleroderma OR systemic lupus erythematosus lupus nephritis) AND (guideline OR consensus OR clinical practice guideline)
PUBMED_RETMAX=30

NCBI_API_KEY=
OPENFDA_API_KEY=
HF_API_TOKEN=
HF_MEDICAL_MODEL=m42-health/Llama3-Med42-8B
USE_STT_MIC_DEFAULT=false
STT_OPENAI_MODEL=gpt-4o-mini
STT_OPENAI_FALLBACK_MODELS=gpt-4o-mini-transcribe,whisper-1
STT_LANGUAGE=en
STT_SAMPLE_RATE=16000
STT_CHUNK_SECONDS=8
STT_NUM_CHUNKS=3
STT_ENABLE_VOICE_STOP=true
STT_VOICE_STOP_KEYWORDS=stop recording,end transcript,stop
STT_SMOKE_TEST=false
```

### Common errors
- `No knowledge docs were loaded` -> set `KB_USE_SEED=true` or enable at least one live-source toggle
- `AccessDeniedException` -> enable model access in AWS Console -> Bedrock -> Model access
- `NoCredentialsError` -> run `aws configure` in terminal
- `ModuleNotFoundError: governance` -> run notebook from project root directory
- `promptfoo not found` -> run `npm install -g promptfoo`
- `STT dependency error` -> run Cell 10a with internet once so optional packages can install
- `OPENAI_API_KEY not set` -> add key to `.env` for API-based STT
- `No microphone transcript captured` -> speak during recording chunks or use Cell 10b smoke mode
- `Runtime integration tests skipped` -> run one consultation first so session artifacts exist in `audit_logs/` and `aura_outputs/`
- `clinical_findings` shows `[Source: Claude Haiku (last resort...)]` -> HF token is missing or Med42 quota exhausted; set `HF_API_TOKEN` in `.env` to enable Tier 2
- `Dynamic replay setup failed: no scrubbed_transcript` -> run Cell 15 again so `scrubbed_transcript` is exported into `*_record.json`